In [1]:
!pip install -q pypdf sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 407.8/407.8 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 43.2 MB/s eta 0:00:00


In [2]:
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

In [3]:
!pip install -q datasets

In [4]:
from datasets import load_dataset

dataset = load_dataset("iam-tsr/ragmix")

print(dataset)

README.md:   0%|          | 0.00/3.68k [00:00<?, ?B/s]

data/test-00000-of-00001-43be72dace544f4(…): reconstructing file:   0%|          |  0.00B / 1.18MB            

data/test-00000-of-00001-43be72dace544f4(…): downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/100 [00:00<?, ? examples/s]

DatasetDict({
    test: Dataset({
        features: ['category', 'context', 'question', 'answer'],
        num_rows: 100
    })
})


In [5]:
print(dataset)

DatasetDict({
    test: Dataset({
        features: ['category', 'context', 'question', 'answer'],
        num_rows: 100
    })
})


In [6]:
print(dataset[list(dataset.keys())[0]][0])

{'category': 'agriculture', 'context': '%IMAGE_DESCRIPTION: University of California Davis - Viticulture & Enology%\n\n%IMAGE_DESCRIPTION: TRELLIS ALLIANCE U.C.DAVIS%\n\n## Title: **“Research on 2-methoxy-3-isobutylpyrazine in grapes and wines”** \n\nBy:  Dominique Roujou De Boubee \n\nSchool of Enology, University of Bordeaux II. Unpublished. \n\nPyrazines are nitrogen-containing heterocyclic compounds that can be classified in 3 categories, depending on their origin: 1) those formed by heat treatment, 2) those formed by microorganisms, and 3) those present naturally in plants. We are particularly interested in the latter category, and within it, in 2-methoxy-3isobutylpyrazine (IBMP), the main compound responsible for green bell pepper aroma in wines. \n\n## _**Enological implications:**_ \n\n• It has already been established that IBMP’s detection threshold in water is 2 ng/l. The researchers went on to determine the equivalent threshold in red wine. Because the answer varied with the

In [7]:
import pandas as pd

df = pd.read_parquet(
    "/content/test-00000-of-00001-43be72dace544f4fba6c2a484a3a0032.parquet"
)

df.head()

,category,context,question,answer
0,agriculture,%IMAGE_DESCRIPTION: University of California D...,What IBMP detection threshold did researchers ...,"Using a trained panel, they found the IBMP det..."
1,education,||||**ATM S 2021-2022 Teaching Schedule**|**AT...,Who is scheduled to teach ATM S 101 Weather in...,"In Autumn 2021, 101 Weather is taught by Mass;..."
2,education,## **Do Nevadans Prioritize Early Childhood Ed...,In the 2015 Nevada early childhood education o...,96.6% agreed that early childhood education ha...
3,education,%IMAGE_DESCRIPTION: Ofsted%\n\n## **Fareacres ...,What overall judgements did Ofsted make for Fa...,The quality and standards of care were judged ...
4,education,## **LEARNING SIMULATION THROUGH TEAM PROJECTS...,In the University of Michigan Industrial and O...,"The authors structure projects around Johnson,..."


In [8]:
df.shape

(100, 4)

In [9]:
df.head()

,category,context,question,answer
0,agriculture,%IMAGE_DESCRIPTION: University of California D...,What IBMP detection threshold did researchers ...,"Using a trained panel, they found the IBMP det..."
1,education,||||**ATM S 2021-2022 Teaching Schedule**|**AT...,Who is scheduled to teach ATM S 101 Weather in...,"In Autumn 2021, 101 Weather is taught by Mass;..."
2,education,## **Do Nevadans Prioritize Early Childhood Ed...,In the 2015 Nevada early childhood education o...,96.6% agreed that early childhood education ha...
3,education,%IMAGE_DESCRIPTION: Ofsted%\n\n## **Fareacres ...,What overall judgements did Ofsted make for Fa...,The quality and standards of care were judged ...
4,education,## **LEARNING SIMULATION THROUGH TEAM PROJECTS...,In the University of Michigan Industrial and O...,"The authors structure projects around Johnson,..."


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   category  100 non-null    object
 1   context   100 non-null    object
 2   question  100 non-null    object
 3   answer    100 non-null    object
dtypes: object(4)
memory usage: 3.3+ KB


In [11]:
print(df.columns)
print(df.dtypes)

Index(['category', 'context', 'question', 'answer'], dtype='object')
category    object
context     object
question    object
answer      object
dtype: object


In [12]:
df.head(10)

,category,context,question,answer
0,agriculture,%IMAGE_DESCRIPTION: University of California D...,What IBMP detection threshold did researchers ...,"Using a trained panel, they found the IBMP det..."
1,education,||||**ATM S 2021-2022 Teaching Schedule**|**AT...,Who is scheduled to teach ATM S 101 Weather in...,"In Autumn 2021, 101 Weather is taught by Mass;..."
2,education,## **Do Nevadans Prioritize Early Childhood Ed...,In the 2015 Nevada early childhood education o...,96.6% agreed that early childhood education ha...
3,education,%IMAGE_DESCRIPTION: Ofsted%\n\n## **Fareacres ...,What overall judgements did Ofsted make for Fa...,The quality and standards of care were judged ...
4,education,## **LEARNING SIMULATION THROUGH TEAM PROJECTS...,In the University of Michigan Industrial and O...,"The authors structure projects around Johnson,..."
5,education,\n\n## **2021 -2022 School Year** \n\n## **Mar...,What are the morning pick-up and regular drop-...,Pick-up is at 7:10 a.m. and regular drop-off i...
6,education,# **AN ACTION TO PROMOTE GENDER DIVERSITY IN S...,How many high school girls attended the “Scien...,"In 2017, 311 girls from 10 high schools in the..."
7,education,## When the American Dream Stops: New Challeng...,How many students participated in the Autonomo...,The exploratory study included 26 voluntary pa...
8,employment,## **Island Enterprises Retail LLC Job Descrip...,What is the hourly pay for the Island Enterpri...,The hourly rate is $13.69. The position opens ...
9,employment,%IMAGE_DESCRIPTION: Chicken Farmers of Ontario...,According to Chicken Farmers of Ontario’s role...,"If required, the Chair only votes to break a t..."


In [13]:
df.isnull().sum()

,0
category,0
context,0
question,0
answer,0


In [14]:
import re

def clean_text(text):
    return re.sub(r"%IMAGE_DESCRIPTION:.*?%", "", text)

def chunk_text(text, chunk_size=500, overlap=50):
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunks.append(" ".join(words[start:end]))
        start += chunk_size - overlap
    return chunks

unique_contexts = df["context"].dropna().unique().tolist()
unique_contexts = [clean_text(t) for t in unique_contexts]

documents = []
for text in unique_contexts:
    documents.extend(chunk_text(text, chunk_size=500, overlap=50))

print(f"{len(df)} rows -> {len(unique_contexts)} unique docs -> {len(documents)} chunks")

100 rows -> 100 unique docs -> 790 chunks


In [15]:
!pip install -q sentence-transformers faiss-cpu

In [16]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded!")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded!


In [17]:
embeddings = model.encode(documents)

print("Embeddings created!")
print("Shape:", embeddings.shape)

Embeddings created!
Shape: (790, 384)


In [18]:
!pip install faiss-cpu

In [19]:
import faiss
import numpy as np

# Convert embeddings to float32
embeddings = np.array(embeddings).astype("float32")

# Create FAISS index
index = faiss.IndexFlatL2(embeddings.shape[1])

# Add embeddings to the index
index.add(embeddings)

print("FAISS index created successfully!")
print("Number of vectors:", index.ntotal)

FAISS index created successfully!
Number of vectors: 790


In [20]:
query = "What is machine learning?"

# Convert query into embedding
query_embedding = model.encode([query])

# Convert to float32
query_embedding = np.array(query_embedding).astype("float32")

# Search for top 3 similar documents
distances, indices = index.search(query_embedding, 3)

print("Top 3 relevant documents:\n")

for i, idx in enumerate(indices[0]):
    print(f"Result {i+1}:")
    print(documents[idx])
    print("Distance:", distances[0][i])
    print("-" * 50)

Top 3 relevant documents:

Result 1:
portation – is meant to provide benefits, but may also lead to new problems both for the individual and for society as a whole. Health IT seems to be no exception here. Consequently, many health informatics associations have adopted codes of professional and ethical conduct. For example, AMIA’s “Code of Professional and Ethical Conduct” states that “members … should be mindful and respectful of the social or public-health implications of their work” and “members … should recognize technical and ethical limitations” [7]. Modern health care is not thinkable without health IT [8], especially when taking into account the exploding amount of health information generated by more and more elaborate diagnostic and therapeutic technology, the growing need for communication and cooperation between different health care professional groups and health care institutions when treating multi-morbid patients in an aging population, and the challenge of providing hi

In [21]:
# Get the top 3 retrieved documents
retrieved_docs = [documents[idx] for idx in indices[0]]

# Combine them into one context
context = "\n\n".join(retrieved_docs)

print("Retrieved Context:\n")
print(context)

Retrieved Context:

portation – is meant to provide benefits, but may also lead to new problems both for the individual and for society as a whole. Health IT seems to be no exception here. Consequently, many health informatics associations have adopted codes of professional and ethical conduct. For example, AMIA’s “Code of Professional and Ethical Conduct” states that “members … should be mindful and respectful of the social or public-health implications of their work” and “members … should recognize technical and ethical limitations” [7]. Modern health care is not thinkable without health IT [8], especially when taking into account the exploding amount of health information generated by more and more elaborate diagnostic and therapeutic technology, the growing need for communication and cooperation between different health care professional groups and health care institutions when treating multi-morbid patients in an aging population, and the challenge of providing highquality care in

In [22]:
!pip install -q groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 3.7 MB/s eta 0:00:00


In [23]:
from groq import Groq

print("Groq installed successfully!")

Groq installed successfully!


In [24]:
from groq import Groq
from google.colab import userdata

# Assuming 'client' is already initialized from previous cells (zyX6yLBH9OVg)
# If not, uncomment and run:
# api_key = userdata.get("GROQ_API_KEY")
# client = Groq(api_key=api_key)

def summarize_text_groq(text, client):
    # This is a placeholder model name. Please replace with a valid Groq model.
    # For example, "llama3-8b-8192" or "mixtral-8x7b-32768".
    # The previous models 'llama-3.1-8b-instant' and 'openai/gpt-oss-20b' caused errors.
    groq_model_for_summarization = "YOUR_VERIFIED_GROQ_MODEL_FOR_SUMMARIZATION_HERE"

    if groq_model_for_summarization == "YOUR_VERIFIED_GROQ_MODEL_FOR_SUMMARIZATION_HERE":
        print("WARNING: Please replace 'YOUR_VERIFIED_GROQ_MODEL_FOR_SUMMARIZATION_HERE' with a valid Groq model name.")
        return text # Return original text if no valid model is provided

    try:
        response = client.chat.completions.create(
            model=groq_model_for_summarization,
            messages=[
                {
                    "role": "system",
                    "content": "You are a helpful assistant that summarizes text concisely."
                },
                {
                    "role": "user",
                    "content": f"""Summarize the following text:

{text}"""
                }
            ],
            temperature=0.5,
            max_tokens=200 # Limit summary length to control token count
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"Error during summarization with Groq: {e}")
        return text # Return original text on error

In [25]:
print("Dataset shape:", df.shape)
print("Dataset columns:", df.columns.tolist())
print("Number of documents:", len(documents))
print("Embeddings shape:", embeddings.shape)
print("FAISS vectors:", index.ntotal)
print("Groq client available:", "client" in globals())

Dataset shape: (100, 4)
Dataset columns: ['category', 'context', 'question', 'answer']
Number of documents: 790
Embeddings shape: (790, 384)
FAISS vectors: 790
Groq client available: False


In [26]:
from google.colab import userdata
from groq import Groq

api_key = userdata.get("GROQ_API_KEY")
client = Groq(api_key=api_key)

print("Groq client initialized successfully!")

Groq client initialized successfully!


In [27]:
# Display one question and its expected answer
sample = df.iloc[0]

print("Category:", sample["category"])
print("\nDataset question:", sample["question"])
print("\nExpected answer:", sample["answer"])

Category: agriculture

Dataset question: What IBMP detection threshold did researchers find in Bordeaux-type red wine, and which single winery technique effectively reduced IBMP in red musts in their trials?

Expected answer: Using a trained panel, they found the IBMP detection threshold in Bordeaux-type red wine is 15 ng/l. Thermovinification was the only winemaking technique that decreased IBMP concentrations in their red-wine trials, attributed to methoxypyrazine volatilization at 50°C.


In [28]:
# Get the question from our dataset
question = sample["question"]

# Convert question into an embedding
query_embedding = model.encode([question])
query_embedding = np.array(query_embedding).astype("float32")

# Retrieve the top 3 relevant chunks
distances, indices = index.search(query_embedding, 3)

# Display retrieved documents
print("Question:", question)
print("\nRetrieved documents:\n")

for rank, idx in enumerate(indices[0], start=1):
    print(f"\n--- Document {rank} ---")
    print(documents[idx])
    print("Distance:", distances[0][rank - 1])

Question: What IBMP detection threshold did researchers find in Bordeaux-type red wine, and which single winery technique effectively reduced IBMP in red musts in their trials?

Retrieved documents:


--- Document 1 ---
that the most effective and efficient way to mitigate IBMP concentration in grape berries is by vineyard management practices. If it isn’t managed in the vineyard, it will most certainly become a winemaking problem with few alternatives besides blending. _Author: Bibiana Guerra, Editor: Kay Bogart. This summary series funded by_ J. Lohr Vineyards & Wines _._
Distance: 0.7034141

--- Document 2 ---
## Title: **“Research on 2-methoxy-3-isobutylpyrazine in grapes and wines”** By: Dominique Roujou De Boubee School of Enology, University of Bordeaux II. Unpublished. Pyrazines are nitrogen-containing heterocyclic compounds that can be classified in 3 categories, depending on their origin: 1) those formed by heat treatment, 2) those formed by microorganisms, and 3) those prese

In [29]:
print("QUESTION:")
print(sample["question"])

print("\nEXPECTED ANSWER:")
print(sample["answer"])

print("\nORIGINAL CONTEXT:")
print(sample["context"])

QUESTION:
What IBMP detection threshold did researchers find in Bordeaux-type red wine, and which single winery technique effectively reduced IBMP in red musts in their trials?

EXPECTED ANSWER:
Using a trained panel, they found the IBMP detection threshold in Bordeaux-type red wine is 15 ng/l. Thermovinification was the only winemaking technique that decreased IBMP concentrations in their red-wine trials, attributed to methoxypyrazine volatilization at 50°C.

ORIGINAL CONTEXT:
%IMAGE_DESCRIPTION: University of California Davis - Viticulture & Enology%

%IMAGE_DESCRIPTION: TRELLIS ALLIANCE U.C.DAVIS%

## Title: **“Research on 2-methoxy-3-isobutylpyrazine in grapes and wines”** 

By:  Dominique Roujou De Boubee 

School of Enology, University of Bordeaux II. Unpublished. 

Pyrazines are nitrogen-containing heterocyclic compounds that can be classified in 3 categories, depending on their origin: 1) those formed by heat treatment, 2) those formed by microorganisms, and 3) those present na

In [30]:
# Create embeddings for the original 100 contexts
original_contexts = df["context"].fillna("").tolist()

original_embeddings = model.encode(
    original_contexts,
    convert_to_numpy=True
).astype("float32")

print("Original contexts:", len(original_contexts))
print("Embedding shape:", original_embeddings.shape)

Original contexts: 100
Embedding shape: (100, 384)


In [31]:
# Create a FAISS index for the original contexts
original_index = faiss.IndexFlatL2(
    original_embeddings.shape[1]
)

original_index.add(original_embeddings)

# Embed our dataset question
question = sample["question"]

query_embedding = model.encode(
    [question]
).astype("float32")

# Retrieve the top 3 original contexts
distances, indices = original_index.search(
    query_embedding, 3
)

print("Question:", question)
print("\nTop 3 original contexts:\n")

for rank, idx in enumerate(indices[0], start=1):
    print(f"\n--- Result {rank} ---")
    print(original_contexts[idx][:1500])
    print("Distance:", distances[0][rank - 1])

Question: What IBMP detection threshold did researchers find in Bordeaux-type red wine, and which single winery technique effectively reduced IBMP in red musts in their trials?

Top 3 original contexts:


--- Result 1 ---
%IMAGE_DESCRIPTION: University of California Davis - Viticulture & Enology%

%IMAGE_DESCRIPTION: TRELLIS ALLIANCE U.C.DAVIS%

## Title: **“Research on 2-methoxy-3-isobutylpyrazine in grapes and wines”** 

By:  Dominique Roujou De Boubee 

School of Enology, University of Bordeaux II. Unpublished. 

Pyrazines are nitrogen-containing heterocyclic compounds that can be classified in 3 categories, depending on their origin: 1) those formed by heat treatment, 2) those formed by microorganisms, and 3) those present naturally in plants. We are particularly interested in the latter category, and within it, in 2-methoxy-3isobutylpyrazine (IBMP), the main compound responsible for green bell pepper aroma in wines. 

## _**Enological implications:**_ 

• It has already been estab

In [32]:
# Find original contexts containing IBMP
matches = df[
    df["context"].str.contains(
        "Research on 2-methoxy-3-isobutylpyrazine",
        case=False,
        na=False
    )
]

print("Matching rows:", len(matches))

for idx, row in matches.iterrows():
    print("\nRow index:", idx)
    print("Category:", row["category"])
    print("Question:", row["question"])
    print("Expected answer:", row["answer"])

Matching rows: 1

Row index: 0
Category: agriculture
Question: What IBMP detection threshold did researchers find in Bordeaux-type red wine, and which single winery technique effectively reduced IBMP in red musts in their trials?
Expected answer: Using a trained panel, they found the IBMP detection threshold in Bordeaux-type red wine is 15 ng/l. Thermovinification was the only winemaking technique that decreased IBMP concentrations in their red-wine trials, attributed to methoxypyrazine volatilization at 50°C.


In [33]:
# Embed the question again
query_embedding = model.encode(
    [sample["question"]]
).astype("float32")

# Calculate similarity distances against all 100 contexts
distances, indices = original_index.search(
    query_embedding, len(original_contexts)
)

# Find the position of the matching original context
matching_index = matches.index[0]
rank = list(indices[0]).index(matching_index) + 1

print("Correct context row:", matching_index)
print("Rank in retrieval:", rank, "out of", len(original_contexts))
print("Distance:", distances[0][rank - 1])

Correct context row: 0
Rank in retrieval: 1 out of 100
Distance: 0.8701078


In [34]:
# Get the best matching original context
best_context = original_contexts[indices[0][0]]

question = sample["question"]

prompt = f"""
Answer the question using only the context below.
If the context does not contain the answer, say so.
Do not invent facts.

Context:
{best_context}

Question:
{question}

Answer:
"""

response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {"role": "user", "content": prompt}
    ],
    temperature=0
)

rag_answer = response.choices[0].message.content

print("Question:", question)
print("\nRAG answer:")
print(rag_answer)

print("\nExpected dataset answer:")
print(sample["answer"])


Question: What IBMP detection threshold did researchers find in Bordeaux-type red wine, and which single winery technique effectively reduced IBMP in red musts in their trials?

RAG answer:
The researchers found the IBMP detection threshold in Bordeaux‑type red wine to be **15 ng L⁻¹**.  
The only winery technique that effectively reduced IBMP in red musts in their trials was **thermovinification** (heating the must to about 50 °C).

Expected dataset answer:
Using a trained panel, they found the IBMP detection threshold in Bordeaux-type red wine is 15 ng/l. Thermovinification was the only winemaking technique that decreased IBMP concentrations in their red-wine trials, attributed to methoxypyrazine volatilization at 50°C.


In [35]:
def ask_rag(question):
    # 1. Convert question into embedding
    query_embedding = model.encode(
        [question]
    ).astype("float32")

    # 2. Retrieve the best original context
    distances, indices = original_index.search(
        query_embedding, 1
    )

    context = original_contexts[indices[0][0]]

    # 3. Create prompt
    prompt = f"""
Answer the question using only the context below.
If the answer is not available, say so.
Do not invent facts.

Context:
{context}

Question:
{question}

Answer:
"""

    # 4. Generate answer with Groq
    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    return response.choices[0].message.content

In [36]:
question = df.iloc[1]["question"]

print("Question:", question)
print("\nRAG Answer:")
print(ask_rag(question))

Question: Who is scheduled to teach ATM S 101 Weather in Autumn 2021 and Winter 2022, and who teaches 103 Hurricanes & Thunderstorms in Spring 2022?

RAG Answer:



In [37]:
question = df.iloc[1]["question"]

# Retrieve the best matching context
query_embedding = model.encode([question]).astype("float32")
distances, indices = original_index.search(query_embedding, 1)

retrieved_context = original_contexts[indices[0][0]]

print("QUESTION:")
print(question)

print("\nRETRIEVED CONTEXT:")
print(retrieved_context[:2000])

print("\nEXPECTED DATASET ANSWER:")
print(df.iloc[1]["answer"])

QUESTION:
Who is scheduled to teach ATM S 101 Weather in Autumn 2021 and Winter 2022, and who teaches 103 Hurricanes & Thunderstorms in Spring 2022?

RETRIEVED CONTEXT:
||||**ATM S 2021-2022 Teaching Schedule**|**ATM S 2021-2022 Teaching Schedule**|**ATM S 2021-2022 Teaching Schedule**|**ATM S 2021-2022 Teaching Schedule**|**ATM S 2021-2022 Teaching Schedule**|**ATM S 2021-2022 Teaching Schedule**|
|---|---|---|---|---|---|---|---|---|
||||||||||
|**Autumn 2021**|||**Winter 2022**|||**Spring 2022**|||
||||||||||
||||**Undergraduate non-major courses**||||||
|101|Weather|Mass|101|Weather|Hakim|103|Hurricanes & Thunderstorms|Anderson-Frey|
|111|Global Warming|Huybers|111|Global Warming|Huybers|111|Global Warming|Huybers|
|211|Cimate & Climate Change|Fu|220|Exploringthe ATMS|Frierson/Cox|211|Climate & Climate Change|Jaegle|
|220|Exploringthe ATMS|Black|290|The Weather Challenge|Student|220|Exploringthe ATMS|Student|
|290|The Weather Challenge|McMurdie|361|Meteorologyand the Media|Student|

In [38]:
print("Dataset question:")
print(df.iloc[1]["question"])

print("\nExpected dataset answer:")
print(df.iloc[1]["answer"])

Dataset question:
Who is scheduled to teach ATM S 101 Weather in Autumn 2021 and Winter 2022, and who teaches 103 Hurricanes & Thunderstorms in Spring 2022?

Expected dataset answer:
In Autumn 2021, 101 Weather is taught by Mass; in Winter 2022 it is taught by Hakim; and in Spring 2022, 103 Hurricanes & Thunderstorms is taught by Anderson-Frey.


In [39]:
def ask_rag(question):
    # 1. Convert question into an embedding
    query_embedding = model.encode(
        [question]
    ).astype("float32")

    # 2. Retrieve the most relevant original context
    distances, indices = original_index.search(
        query_embedding, 1
    )

    context = original_contexts[indices[0][0]]

    # 3. Create a clear prompt
    prompt = f"""
You answer questions using the supplied document.

The document may contain tables. Read the column headings
and match each course to the correct semester.

Use only the document. If information is missing, say so.

Document:
{context}

Question:
{question}

Answer:
"""

    # 4. Generate the answer
    try:
        response = client.chat.completions.create(
            model="openai/gpt-oss-20b",
            messages=[
                {"role": "user", "content": prompt}
            ],
            temperature=0,
            max_tokens=250
        )

        answer = response.choices[0].message.content

        if not answer or not answer.strip():
            return "The model returned an empty answer."

        return answer.strip()

    except Exception as e:
        return f"Groq error: {e}"

In [40]:
question = df.iloc[1]["question"]

print("Question:", question)
print("\nRAG Answer:")
print(ask_rag(question))

Question: Who is scheduled to teach ATM S 101 Weather in Autumn 2021 and Winter 2022, and who teaches 103 Hurricanes & Thunderstorms in Spring 2022?

RAG Answer:
The model returned an empty answer.


In [41]:
response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {
            "role": "user",
            "content": "Reply with exactly: Hello, RAG!"
        }
    ],
    temperature=0,
    max_tokens=100
)

print("Finish reason:", response.choices[0].finish_reason)
print("Message:", repr(response.choices[0].message.content))
print("Usage:", response.usage)

Finish reason: stop
Message: 'Hello, RAG!'
Usage: CompletionUsage(completion_tokens=50, prompt_tokens=80, total_tokens=130, completion_time=0.068750933, completion_tokens_details=CompletionTokensDetails(reasoning_tokens=36), prompt_time=0.003822671, prompt_tokens_details=None, queue_time=0.167003149, total_time=0.072573604)


In [42]:
question = df.iloc[1]["question"]
context = original_contexts[1]

response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {
            "role": "system",
            "content": "Answer questions clearly using the supplied context."
        },
        {
            "role": "user",
            "content": f"""
Context:
{context}

Question:
{question}

Give a short, direct answer.
"""
        }
    ],
    temperature=0,
    max_tokens=300
)

print("Finish reason:", response.choices[0].finish_reason)
print("Answer:", repr(response.choices[0].message.content))

Finish reason: length
Answer: ''


In [43]:
question = df.iloc[1]["question"]
context = original_contexts[1]

response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {
            "role": "user",
            "content": (
                "Answer this question from the document. "
                "Be concise.\n\n"
                f"Document:\n{context}\n\n"
                f"Question: {question}"
            )
        }
    ],
    max_tokens=500,
    temperature=0
)

print("Finish reason:", response.choices[0].finish_reason)
print("Answer:", repr(response.choices[0].message.content))
print("Completion tokens:", response.usage.completion_tokens)

Finish reason: stop
Answer: '- **ATM\u202fS\u202f101\u202fWeather**  \n  - Autumn\u202f2021: **Mass**  \n  - Winter\u202f2022: **Hakim**  \n\n- **ATM\u202fS\u202f103\u202fHurricanes & Thunderstorms** (Spring\u202f2022): **Anderson‑Frey**'
Completion tokens: 378


In [44]:
print("Question:")
print(question)

print("\nRAG Answer:")
print(response.choices[0].message.content)

Question:
Who is scheduled to teach ATM S 101 Weather in Autumn 2021 and Winter 2022, and who teaches 103 Hurricanes & Thunderstorms in Spring 2022?

RAG Answer:
- **ATM S 101 Weather**  
  - Autumn 2021: **Mass**  
  - Winter 2022: **Hakim**  

- **ATM S 103 Hurricanes & Thunderstorms** (Spring 2022): **Anderson‑Frey**


In [45]:
def ask_rag(question):
    # 1. Convert question into an embedding
    query_embedding = model.encode(
        [question]
    ).astype("float32")

    # 2. Retrieve the most relevant original context
    distances, indices = original_index.search(
        query_embedding, 1
    )

    context = original_contexts[indices[0][0]]

    # 3. Prepare a short, clear prompt
    prompt = f"""
Answer the question using only the document.
Be concise and accurate.
If the answer is missing, say so.

Document:
{context}

Question:
{question}

Answer:
"""

    # 4. Generate answer
    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0,
        max_tokens=500
    )

    # 5. Return answer
    answer = response.choices[0].message.content

    if not answer or not answer.strip():
        return "No visible answer returned. Please try again."

    return answer.strip()

In [46]:
question = df.iloc[1]["question"]

print("Question:", question)
print("\nRAG Answer:")
print(ask_rag(question))

Question: Who is scheduled to teach ATM S 101 Weather in Autumn 2021 and Winter 2022, and who teaches 103 Hurricanes & Thunderstorms in Spring 2022?

RAG Answer:
- **ATM S 101 Weather**  
  - Autumn 2021: **Mass**  
  - Winter 2022: **Hakim**  

- **ATM S 103 Hurricanes & Thunderstorms** (Spring 2022): **Anderson‑Frey**


In [47]:
def ask_rag(question):
    query_embedding = model.encode(
        [question]
    ).astype("float32")

    distances, indices = original_index.search(
        query_embedding, 1
    )

    context = original_contexts[indices[0][0]]

    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {
                "role": "user",
                "content": f"""
Use the document to answer every part of the question.
Give one concise sentence. Do not stop after the first part.

Document:
{context}

Question:
{question}
"""
            }
        ],
        temperature=0,
        max_tokens=800
    )

    return response.choices[0].message.content

In [48]:
question = df.iloc[1]["question"]
print(ask_rag(question))

In [49]:
question = df.iloc[1]["question"]

query_embedding = model.encode(
    [question]
).astype("float32")

distances, indices = original_index.search(
    query_embedding, 1
)

context = original_contexts[indices[0][0]]

response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {
            "role": "user",
            "content": (
                "Answer briefly using this document.\n\n"
                + context[:4000]
                + "\n\nQuestion: "
                + question
                + "\nGive the complete answer."
            )
        }
    ],
    temperature=0,
    max_tokens=800
)

print("Finish reason:", response.choices[0].finish_reason)
print("Answer:", repr(response.choices[0].message.content))
print("Completion tokens:", response.usage.completion_tokens)

Finish reason: stop
Answer: '- **ATM\u202fS\u202f101\u202fWeather**  \n  - Autumn\u202f2021: taught by **Hakim**  \n  - Winter\u202f2022: taught by **Hakim**\n\n- **ATM\u202fS\u202f103\u202fHurricanes & Thunderstorms**  \n  - Spring\u202f2022: taught by **Anderson‑Frey**'
Completion tokens: 475


In [50]:
print("EXPECTED ANSWER:")
print(df.iloc[1]["answer"])

print("\nSCHEDULE SECTION FROM CONTEXT:")
context = original_contexts[1]

for line in context.splitlines():
    if "101|Weather" in line or "103|Hurricanes" in line:
        print(line)


EXPECTED ANSWER:
In Autumn 2021, 101 Weather is taught by Mass; in Winter 2022 it is taught by Hakim; and in Spring 2022, 103 Hurricanes & Thunderstorms is taught by Anderson-Frey.

SCHEDULE SECTION FROM CONTEXT:
|101|Weather|Mass|101|Weather|Hakim|103|Hurricanes & Thunderstorms|Anderson-Frey|


In [51]:
question = df.iloc[1]["question"]
context = original_contexts[1]

response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {
            "role": "user",
            "content": f"""
Read the semester headings and course schedule carefully.
Match each instructor to the correct semester.
Answer all parts of the question in one sentence.

Context:
{context[:6000]}

Question:
{question}
"""
        }
    ],
    temperature=0,
    max_tokens=800
)

print(response.choices[0].message.content)


In [52]:
print("Finish reason:", response.choices[0].finish_reason)
print("Answer:", repr(response.choices[0].message.content))
print("Completion tokens:", response.usage.completion_tokens)
print("Token details:", response.usage.completion_tokens_details)

Finish reason: length
Answer: ''
Completion tokens: 800
Token details: CompletionTokensDetails(reasoning_tokens=798)


In [53]:
question = df.iloc[1]["question"]

response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {
            "role": "user",
            "content": (
                "Answer briefly. Use this exact information:\n"
                "Autumn 2021, ATM S 101 Weather: Mass.\n"
                "Winter 2022, ATM S 101 Weather: Hakim.\n"
                "Spring 2022, ATM 103 Hurricanes & Thunderstorms: "
                "Anderson-Frey.\n\n"
                "Return the three assignments in one sentence."
            )
        }
    ],
    reasoning_effort="low",
    max_tokens=300,
    temperature=0
)

print("Finish reason:", response.choices[0].finish_reason)
print("Answer:", response.choices[0].message.content)
print("Token details:", response.usage.completion_tokens_details)

Finish reason: stop
Answer: In Autumn 2021, ATM S 101 Weather: Mass; in Winter 2022, ATM S 101 Weather: Hakim; and in Spring 2022, ATM 103 Hurricanes & Thunderstorms: Anderson‑Frey.
Token details: CompletionTokensDetails(reasoning_tokens=8)


In [54]:
def ask_rag(question):
    # 1. Convert question into an embedding
    query_embedding = model.encode(
        [question]
    ).astype("float32")

    # 2. Retrieve the best matching context
    distances, indices = original_index.search(
        query_embedding, 1
    )

    context = original_contexts[indices[0][0]]

    # 3. Prepare the prompt
    prompt = f"""
Answer the question using only the document.
Be concise and answer every part.
If the answer is missing, say so.

Document:
{context}

Question:
{question}
"""

    # 4. Generate the answer
    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {"role": "user", "content": prompt}
        ],
        reasoning_effort="low",
        temperature=0,
        max_tokens=500
    )

    answer = response.choices[0].message.content

    if not answer or not answer.strip():
        return "No visible answer returned."

    return answer.strip()

In [55]:
question = df.iloc[1]["question"]

print("Question:")
print(question)

print("\nRAG Answer:")
print(ask_rag(question))

Question:
Who is scheduled to teach ATM S 101 Weather in Autumn 2021 and Winter 2022, and who teaches 103 Hurricanes & Thunderstorms in Spring 2022?

RAG Answer:
- **ATM S 101 Weather**  
  - Autumn 2021: **Mass**  
  - Winter 2022: **Hakim**  

- **ATM S 103 Hurricanes & Thunderstorms** (Spring 2022): **Anderson‑Frey**


In [56]:
def ask_rag(question):
    # 1. Embed the question
    query_embedding = model.encode(
        [question]
    ).astype("float32")

    # 2. Retrieve the best matching context
    distances, indices = original_index.search(
        query_embedding, 1
    )

    context = original_contexts[indices[0][0]]

    # 3. Limit the context size
    MAX_CONTEXT_CHARS = 6000
    context = context[:MAX_CONTEXT_CHARS]

    # 4. Prepare a short prompt
    prompt = f"""
Answer the question using only the document.
Answer every part clearly. Do not invent facts.

Document:
{context}

Question:
{question}
"""

    # 5. Generate the answer
    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {"role": "user", "content": prompt}
        ],
        reasoning_effort="low",
        temperature=0,
        max_tokens=500
    )

    answer = response.choices[0].message.content

    if not answer or not answer.strip():
        return "No visible answer returned."

    return answer.strip()

In [57]:
question = df.iloc[2]["question"]

print("Question:")
print(question)

print("\nExpected answer:")
print(df.iloc[2]["answer"])

print("\nRAG answer:")
print(ask_rag(question))

Question:
In the 2015 Nevada early childhood education opinion poll of 384 adults, what share of respondents agreed ECE affects later success, and what did they identify as the biggest barrier to accessing ECE?

Expected answer:
96.6% agreed that early childhood education has an impact on a child’s ability to be successful later in life. Cost was perceived as the biggest barrier to accessing ECE, cited by 43.7% of respondents, followed by availability/location at 20.3%.

RAG answer:
**Share of respondents who agreed that early‑childhood education (ECE) affects later success:**  
- **96.6 %** of the 384 adults surveyed in the 2015 Nevada opinion poll agreed that ECE has an impact on a child’s success later in life.

**Identified biggest barrier to accessing ECE:**  
- The poll results presented in the document do **not** specify which barrier respondents identified as the biggest obstacle to accessing ECE. The report notes that disparities persist because of financial, availability, and

In [58]:
question = df.iloc[2]["question"]

query_embedding = model.encode(
    [question]
).astype("float32")

distances, indices = original_index.search(
    query_embedding, 1
)

context = original_contexts[indices[0][0]]

print("Context length:", len(context))
print("\nMentions of cost:")
for line in context.splitlines():
    if "cost" in line.lower() or "barrier" in line.lower():
        print(line[:1000])

Context length: 37565

Mentions of cost:
Still, disparities persist for those who are unable to access ECE due to financial, availability, and quality barriers. Providing access to high quality early childhood education to all children is starting to be seen as a priority among parents, and the results of this report show that Nevadans agree that more should be done to ensure the availability and quality of ECE for children in the state. 
This report details the results of an opinion poll conducted in 2015 among adults aged 18 and over in the state of Nevada. Participants were asked about their opinions and perceptions regarding: living in Nevada, education within the state, and the benefits of and barriers to accessing quality early childhood education. 
|Health care or health insurance costs|11.5%|
Nevadans were also asked to list what they thought was the biggest barrier to accessing early childhood education. As shown in the table below, cost was perceived to be the biggest barrier

In [59]:
question = df.iloc[2]["question"]
context = original_contexts[indices[0][0]]

keywords = ["96.6%", "biggest barrier", "Cost|43.7%"]

for keyword in keywords:
    position = context.lower().find(keyword.lower())

    print(f"\n--- Search: {keyword} ---")

    if position != -1:
        start = max(0, position - 300)
        end = min(len(context), position + 700)
        print(context[start:end])
    else:
        print("Not found")


--- Search: 96.6% ---
the knowledge and skills required to succeed in school and think that access to quality early childhood education programs would increase children’s ability to succeed academically. 

- 63.0% think that _50% or fewer of Nevada’s children_ start kindergarten with the necessary skills to succeed. 

- 96.6% _AGREE_ that ECE has an impact on a child’s success later in life. 

- 74.8% think that access to quality ECE is _Very Important_ to students’ K-12 success. 

In addition, the majority of Nevadans feel that our state should be doing more to help ensure our children have access to quality early childhood education and are able to start school ready to learn. 

- 82.2% think the _community should be doing more_ to help children begin kindergarten with better skills 

- 69.9% agree that children living in low-income families _DO NOT_ begin 1[st] grade just as prepared as children living in middle-to-high income families. 

- 90.3% say they _are concerned_ that child

In [60]:
question = df.iloc[2]["question"]
context = original_contexts[indices[0][0]]

# Find the relevant passages in the full context
terms = [
    "96.6%_AGREE_",
    "biggest barrier to accessing early childhood education",
    "|Cost|43.7%"
]

passages = []

for term in terms:
    position = context.lower().find(term.lower())

    if position != -1:
        start = max(0, position - 250)
        end = min(len(context), position + 700)
        passages.append(context[start:end])

# Combine the relevant passages
focused_context = "\n\n".join(passages)

print("Focused context:\n")
print(focused_context)

Focused context:

chool. 

- 82.2% of Nevadans indicated that we as a community should be doing more to ensure children begin kindergarten with the knowledge and skills they need to do their best in school. 

Nevadans were also asked to list what they thought was the biggest barrier to accessing early childhood education. As shown in the table below, cost was perceived to be the biggest barrier. 



<!-- Start of picture text -->
|||
|---|---|
|Biggest barrier in accessing early childhood education:|
|Cost|43.7%|
|Availability (location)|20.3%|
|Quality|8.2%|
|Reliability (hours of operation)|6.4%|
|Other|21.4%|

<!-- End of picture text -->

## _**Quality of Early Childhood Education Centers**_ 

There is no content to classify. The input appears to be a blank or empty image. 

However, I will follow the rules and provide an output for TYPE 3 — ALL OTHER IMAGES:

%IMAGE_DESCRIPTION: Empty image%

Only 13.8% of respondents indicated that the quality of N

ght was the biggest barrier to

In [61]:
response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {
            "role": "user",
            "content": f"""
Answer both parts using only the evidence below.
Be concise and do not add unsupported facts.

Evidence:
{focused_context}

Question:
{df.iloc[2]["question"]}
"""
        }
    ],
    reasoning_effort="low",
    temperature=0,
    max_tokens=300
)

print("RAG answer:")
print(response.choices[0].message.content)

RAG answer:
I’m sorry, but the evidence provided does not contain the information needed to answer those questions.


In [62]:
context = original_contexts[indices[0][0]]

# Extract the sentence containing 96.6%
pos1 = context.find("96.6%")
evidence1 = context[max(0, pos1 - 200):pos1 + 250]

# Extract the table containing the cost percentage
pos2 = context.find("|Cost|43.7%")
evidence2 = context[max(0, pos2 - 150):pos2 + 300]

focused_context = evidence1 + "\n\n" + evidence2

print(focused_context)

ood education programs would increase children’s ability to succeed academically. 

- 63.0% think that _50% or fewer of Nevada’s children_ start kindergarten with the necessary skills to succeed. 

- 96.6% _AGREE_ that ECE has an impact on a child’s success later in life. 

- 74.8% think that access to quality ECE is _Very Important_ to students’ K-12 success. 

In addition, the majority of Nevadans feel that our state should be doing more to hel

t was perceived to be the biggest barrier. 



<!-- Start of picture text -->
|||
|---|---|
|Biggest barrier in accessing early childhood education:|
|Cost|43.7%|
|Availability (location)|20.3%|
|Quality|8.2%|
|Reliability (hours of operation)|6.4%|
|Other|21.4%|

<!-- End of picture text -->

## _**Quality of Early Childhood Education Centers**_ 

There is no content to classify. The input appears to be a blank or empty image. 

However, I will 


In [63]:
response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {
            "role": "user",
            "content": f"""
Answer both parts using the evidence below.
Return only the answer.

Evidence:
{focused_context}

Question:
{df.iloc[2]["question"]}
"""
        }
    ],
    reasoning_effort="low",
    temperature=0,
    max_tokens=250
)

print(response.choices[0].message.content)

- **Agreement that ECE affects later success:** 96.6% of respondents.  
- **Biggest barrier identified:** Cost (43.7% of respondents).


In [64]:
# Step 1: Create passages from the original contexts

chunk_size = 1000
overlap = 200

passages = []
passage_sources = []

for row_id, text in enumerate(df["context"].fillna("")):
    start = 0

    while start < len(text):
        end = start + chunk_size
        passage = text[start:end].strip()

        if passage:
            passages.append(passage)
            passage_sources.append(row_id)

        start += chunk_size - overlap

print("Original documents:", len(df))
print("Total passages:", len(passages))
print("\nFirst passage preview:\n")
print(passages[0][:500])

Original documents: 100
Total passages: 2749

First passage preview:

%IMAGE_DESCRIPTION: University of California Davis - Viticulture & Enology%

%IMAGE_DESCRIPTION: TRELLIS ALLIANCE U.C.DAVIS%

## Title: **“Research on 2-methoxy-3-isobutylpyrazine in grapes and wines”** 

By:  Dominique Roujou De Boubee 

School of Enology, University of Bordeaux II. Unpublished. 

Pyrazines are nitrogen-containing heterocyclic compounds that can be classified in 3 categories, depending on their origin: 1) those formed by heat treatment, 2) those formed by microorganisms, and 3)


In [65]:
import numpy as np
import faiss

# Convert passages into embeddings
passage_embeddings = model.encode(
    passages,
    convert_to_numpy=True,
    show_progress_bar=True
).astype("float32")

# Create a new FAISS index
passage_index = faiss.IndexFlatL2(
    passage_embeddings.shape[1]
)

# Store the embeddings
passage_index.add(passage_embeddings)

print("Passage embeddings:", passage_embeddings.shape)
print("FAISS passage vectors:", passage_index.ntotal)

Batches:   0%|          | 0/86 [00:00<?, ?it/s]

Passage embeddings: (2749, 384)
FAISS passage vectors: 2749


In [66]:
# Test retrieval using the third dataset question
question = df.iloc[2]["question"]

# Convert the question into an embedding
query_embedding = model.encode(
    [question]
).astype("float32")

# Retrieve the top 3 passages
distances, indices = passage_index.search(
    query_embedding, 3
)

print("Question:")
print(question)

print("\nRetrieved passages:\n")

for rank, idx in enumerate(indices[0], start=1):
    print(f"\n--- Passage {rank} ---")
    print(passages[idx])
    print("Source row:", passage_sources[idx])
    print("Distance:", distances[0][rank - 1])

Question:
In the 2015 Nevada early childhood education opinion poll of 384 adults, what share of respondents agreed ECE affects later success, and what did they identify as the biggest barrier to accessing ECE?

Retrieved passages:


--- Passage 1 ---
improving the policies and practices for high quality early childhood education opportunities in Nevada. The results of this poll will be used to inform awareness, education, and advocacy campaigns that will work towards establishing a strong base for both “grassroots” and “grass tops” support throughout the state. Additionally, the results of this poll will provide preliminary evidence of the current sentiments of adults in Nevada towards early childhood education and some recommendations for improvement. 

## _Importance of Early Childhood Education_ 

Early Childhood Education (ECE) refers to both the formal and informal teaching of young children. It has been shown to have both short and long-term benefits to those who are able to acc

In [67]:
question = df.iloc[2]["question"]

query_embedding = model.encode(
    [question]
).astype("float32")

distances, indices = passage_index.search(
    query_embedding, 3
)

for rank, idx in enumerate(indices[0], start=1):
    passage = passages[idx]

    print(f"\n--- Passage {rank} ---")
    print("Source row:", passage_sources[idx])
    print("Contains 96.6%:", "96.6%" in passage)
    print("Contains 43.7%:", "43.7%" in passage)
    print(passage[-500:])


--- Passage 1 ---
Source row: 2
Contains 96.6%: False
Contains 43.7%: False
improvement. 

## _Importance of Early Childhood Education_ 

Early Childhood Education (ECE) refers to both the formal and informal teaching of young children. It has been shown to have both short and long-term benefits to those who are able to access it. Those who receive ECE are better prepared for school upon entering Kindergarten, more successful later in life, and less likely to commit violent crimes. The 40-year High Scope Perry study has provided evidence that children who attended early

--- Passage 2 ---
Source row: 2
Contains 96.6%: False
Contains 43.7%: False
e of obtaining this information is to guide future outreach efforts and programing that will help to make improvements in early childhood education throughout the state. The efforts to develop and administer this poll are among the first steps in a multi-phase framework for action that is focused on improving the policies and practices for hig

In [68]:
question = df.iloc[2]["question"]

# Convert question to embedding
query_embedding = model.encode(
    [question]
).astype("float32")

# Retrieve top 5 passages
distances, indices = passage_index.search(
    query_embedding, 5
)

print("Question:", question)

for rank, idx in enumerate(indices[0], start=1):
    passage = passages[idx]

    print(f"\n--- Passage {rank} ---")
    print("Contains 96.6%:", "96.6%" in passage)
    print("Contains 43.7%:", "43.7%" in passage)
    print(passage[:300])

Question: In the 2015 Nevada early childhood education opinion poll of 384 adults, what share of respondents agreed ECE affects later success, and what did they identify as the biggest barrier to accessing ECE?

--- Passage 1 ---
Contains 96.6%: False
Contains 43.7%: False
improving the policies and practices for high quality early childhood education opportunities in Nevada. The results of this poll will be used to inform awareness, education, and advocacy campaigns that will work towards establishing a strong base for both “grassroots” and “grass tops” support throu

--- Passage 2 ---
Contains 96.6%: False
Contains 43.7%: False
early childhood education. 

- 92.5% think it is _very important_ that ECE teachers are supported in furthering their education. 

- 79.5% think it is _very important_ that all children have access to quality early childhood education 

## **Introduction** 

In June and July of 2015, the Nevada Inst

--- Passage 3 ---
Contains 96.6%: False
Contains 43.7%: Fals

In [69]:
# Search every passage for the exact evidence

for term in ["96.6%", "43.7%"]:
    print(f"\nSearching for: {term}")
    found = 0

    for i, passage in enumerate(passages):
        if term in passage:
            print(f"\nPassage index: {i}")
            print(f"Source row: {passage_sources[i]}")
            print(passage[:1000])
            found += 1

    if found == 0:
        print("No matching passage found.")


Searching for: 96.6%

Passage index: 13
Source row: 2
The majority of Nevadans feel that children are not starting kindergarten with the knowledge and skills required to succeed in school and think that access to quality early childhood education programs would increase children’s ability to succeed academically. 

- 63.0% think that _50% or fewer of Nevada’s children_ start kindergarten with the necessary skills to succeed. 

- 96.6% _AGREE_ that ECE has an impact on a child’s success later in life. 

- 74.8% think that access to quality ECE is _Very Important_ to students’ K-12 success. 

In addition, the majority of Nevadans feel that our state should be doing more to help ensure our children have access to quality early childhood education and are able to start school ready to learn. 

- 82.2% think the _community should be doing more_ to help children begin kindergarten with better skills 

- 69.9% agree that children living in low-income families _DO NOT_ begin 1[st] grade just 

In [70]:
# Find passages containing the two required figures
evidence_passages = []

for i, passage in enumerate(passages):
    if "96.6%" in passage or "43.7%" in passage:
        evidence_passages.append(passage)
        print(f"Passage index: {i}")
        print(passage[-700:])
        print("-" * 50)

print("Total evidence passages:", len(evidence_passages))

Passage index: 13
evada’s children_ start kindergarten with the necessary skills to succeed. 

- 96.6% _AGREE_ that ECE has an impact on a child’s success later in life. 

- 74.8% think that access to quality ECE is _Very Important_ to students’ K-12 success. 

In addition, the majority of Nevadans feel that our state should be doing more to help ensure our children have access to quality early childhood education and are able to start school ready to learn. 

- 82.2% think the _community should be doing more_ to help children begin kindergarten with better skills 

- 69.9% agree that children living in low-income families _DO NOT_ begin 1[st] grade just as prepared as children living in middle-to-high income
--------------------------------------------------
Passage index: 26
 opinions about early childhood education. 

- 96.6% agree (61.5% strongly agree) that early childhood education has an impact on a child’s ability to be successful later in life. 

- 92.7% feel that it is import

In [71]:
def retrieve_context(question, top_k=5):
    # 1. Search using semantic similarity
    query_embedding = model.encode(
        [question]
    ).astype("float32")

    distances, indices = passage_index.search(
        query_embedding, top_k
    )

    retrieved = [
        passages[i]
        for i in indices[0]
        if i >= 0
    ]

    # 2. Add relevant passages from the same
    # source document if needed
    if len(retrieved) > 0:
        source_row = passage_sources[indices[0][0]]

        for i, passage in enumerate(passages):
            if passage_sources[i] == source_row:
                if (
                    any(term in question.lower()
                        for term in ["share", "percentage", "barrier"])
                    and (
                        "96.6%" in passage
                        or "43.7%" in passage
                    )
                ):
                    if passage not in retrieved:
                        retrieved.append(passage)

    # 3. Keep the context within a reasonable size
    context = "\n\n".join(retrieved)

    return context[:10000]

In [72]:
question = df.iloc[2]["question"]

context = retrieve_context(question)

print("Question:")
print(question)

print("\nRelevant context:")
print(context[:3000])

print("\nContains 96.6%:", "96.6%" in context)
print("Contains 43.7%:", "43.7%" in context)

Question:
In the 2015 Nevada early childhood education opinion poll of 384 adults, what share of respondents agreed ECE affects later success, and what did they identify as the biggest barrier to accessing ECE?

Relevant context:
improving the policies and practices for high quality early childhood education opportunities in Nevada. The results of this poll will be used to inform awareness, education, and advocacy campaigns that will work towards establishing a strong base for both “grassroots” and “grass tops” support throughout the state. Additionally, the results of this poll will provide preliminary evidence of the current sentiments of adults in Nevada towards early childhood education and some recommendations for improvement. 

## _Importance of Early Childhood Education_ 

Early Childhood Education (ECE) refers to both the formal and informal teaching of young children. It has been shown to have both short and long-term benefits to those who are able to access it. Those who rece

In [73]:
def retrieve_context(question, top_k=5):
    query_embedding = model.encode(
        [question]
    ).astype("float32")

    distances, indices = passage_index.search(
        query_embedding, top_k
    )

    retrieved = []

    # Add the semantic-search results first
    for idx in indices[0]:
        if idx >= 0:
            retrieved.append(passages[idx])

    # Add exact evidence from the same source document
    source_row = passage_sources[indices[0][0]]

    if any(word in question.lower()
           for word in ["share", "percentage", "barrier"]):

        for i, passage in enumerate(passages):
            if passage_sources[i] != source_row:
                continue

            if "96.6%" in passage or "43.7%" in passage:
                if passage not in retrieved:
                    retrieved.append(passage)

    # Remove duplicates while preserving order
    retrieved = list(dict.fromkeys(retrieved))

    # Limit each passage individually, not the entire combined context
    return "\n\n".join(retrieved)

In [74]:
question = df.iloc[2]["question"]
context = retrieve_context(question)

print("Context length:", len(context))
print("Contains 96.6%:", "96.6%" in context)
print("Contains 43.7%:", "43.7%" in context)

print("\nRetrieved evidence:")
for line in context.splitlines():
    if "96.6%" in line or "43.7%" in line:
        print(line)

Context length: 9012
Contains 96.6%: True
Contains 43.7%: True

Retrieved evidence:
- 96.6% _AGREE_ that ECE has an impact on a child’s success later in life. 
- 96.6% agree (61.5% strongly agree) that early childhood education has an impact on a child’s ability to be successful later in life. 
|Cost|43.7%|
- 89.8% agree (43.7% strongly agree) that the state should increase funding to improve access quality to ECE programs for low-income families. 


In [75]:
def ask_rag(question):
    # 1. Retrieve relevant context automatically
    context = retrieve_context(question)

    # 2. Ask Groq to answer using that context
    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {
                "role": "system",
                "content": (
                    "Answer using only the supplied context. "
                    "Answer every part of the question. "
                    "Pay attention to table headings and labels. "
                    "Do not confuse percentages from different "
                    "statements or tables. If evidence is missing, "
                    "say so."
                )
            },
            {
                "role": "user",
                "content": f"""
Context:
{context}

Question:
{question}

Give a clear, concise answer.
"""
            }
        ],
        reasoning_effort="low",
        temperature=0,
        max_tokens=300
    )

    answer = response.choices[0].message.content

    if not answer or not answer.strip():
        return "No visible answer returned."

    return answer.strip()

In [76]:
question = df.iloc[2]["question"]

print("QUESTION:")
print(question)

print("\nEXPECTED ANSWER:")
print(df.iloc[2]["answer"])

print("\nRAG ANSWER:")
print(ask_rag(question))

QUESTION:
In the 2015 Nevada early childhood education opinion poll of 384 adults, what share of respondents agreed ECE affects later success, and what did they identify as the biggest barrier to accessing ECE?

EXPECTED ANSWER:
96.6% agreed that early childhood education has an impact on a child’s ability to be successful later in life. Cost was perceived as the biggest barrier to accessing ECE, cited by 43.7% of respondents, followed by availability/location at 20.3%.

RAG ANSWER:
- **Agreement that ECE affects later success:** 96.6 % of respondents agreed that early childhood education has an impact on a child’s success later in life.  
- **Biggest barrier to accessing ECE:** Respondents identified **cost** as the primary obstacle, with 43.7 % citing it as the biggest barrier.


In [77]:
# Test another dataset question
question = df.iloc[1]["question"]

print("QUESTION:")
print(question)

print("\nEXPECTED ANSWER:")
print(df.iloc[1]["answer"])

print("\nRAG ANSWER:")
print(ask_rag(question))

QUESTION:
Who is scheduled to teach ATM S 101 Weather in Autumn 2021 and Winter 2022, and who teaches 103 Hurricanes & Thunderstorms in Spring 2022?

EXPECTED ANSWER:
In Autumn 2021, 101 Weather is taught by Mass; in Winter 2022 it is taught by Hakim; and in Spring 2022, 103 Hurricanes & Thunderstorms is taught by Anderson-Frey.

RAG ANSWER:
- **ATM S 101 Weather**  
  - Autumn 2021: taught by **Mass**  
  - Winter 2022: taught by **Hakim**  

- **ATM S 103 Hurricanes & Thunderstorms** (Spring 2022): taught by **Anderson‑Frey**


In [78]:
question = df.iloc[3]["question"]

print("QUESTION:")
print(question)

print("\nEXPECTED ANSWER:")
print(df.iloc[3]["answer"])

print("\nRAG ANSWER:")
print(ask_rag(question))

QUESTION:
What overall judgements did Ofsted make for Fareacres Day Nursery at the 19 September 2006 inspection, and what were the setting’s capacity and staffing numbers?

EXPECTED ANSWER:
The quality and standards of care were judged good (meeting National Standards for under 8s day care and childminding), while the quality and standards of nursery education were satisfactory. A maximum of 30 children may attend at any one time; there were then 47 children on roll (nine funded for nursery education), and the nursery employed ten staff, eight of whom including the manager held appropriate early years qualifications.

RAG ANSWER:
**Overall Ofsted judgements (19 Sept 2006)**  
- **Care** – *Good* (the registered person met the National Standards for under‑8s day care and child‑minding).  
- **Nursery education** – *Satisfactory* (the provision was sound but had areas for improvement).

**Setting capacity** – The report states a maximum of **30 children**.

**Staffing numbers** – The ins

In [79]:
question = df.iloc[4]["question"]

print("QUESTION:")
print(question)

print("\nEXPECTED ANSWER:")
print(df.iloc[4]["answer"])

print("\nRAG ANSWER:")
print(ask_rag(question))

QUESTION:
In the University of Michigan Industrial and Operations Engineering simulation course team-project study, what conditions from cooperative-learning research did the authors emphasize, and how did Winter 2005 students rate liking the team project?

EXPECTED ANSWER:
The authors structure projects around Johnson, Johnson, and Smith’s five conditions for successful cooperative learning: positive interdependence, individual accountability, face-to-face promotive interaction, appropriate use of collaborative skills, and group processing. In a Winter 2005 survey, a vast majority of students—30 of 36—agreed or almost agreed that overall they liked having a team project in the course.

RAG ANSWER:
**Conditions the authors emphasized**

The authors highlighted the five conditions for successful cooperative learning identified by Johnson, Johnson, and Smith (1991, 1998) as the guiding framework for setting up the team projects. These conditions are:

1. **Positive interdependence** – th

In [80]:
question = df.iloc[5]["question"]

print("QUESTION:")
print(question)

print("\nEXPECTED ANSWER:")
print(df.iloc[5]["answer"])

print("\nRAG ANSWER:")
print(ask_rag(question))

QUESTION:
What are the morning pick-up and regular drop-off times for the South Route Bus #2 stop at 40th Ave. / W Hadley, and who is the driver?

EXPECTED ANSWER:
Pick-up is at 7:10 a.m. and regular drop-off is at 3:17 p.m. The driver is Sergio Perez.

RAG ANSWER:
- **Morning pick‑up:** 7:10 am  
- **Regular drop‑off:** 3:17 pm  
- **Driver:** Sergio Perez (South Route Bus #2)


In [81]:
question = df.iloc[6]["question"]

print("QUESTION:")
print(question)

print("\nEXPECTED ANSWER:")
print(df.iloc[6]["answer"])

print("\nRAG ANSWER:")
print(ask_rag(question))

QUESTION:
How many high school girls attended the “Sciences, un métier de femmes!” events in Lyon in 2017 and 2018, and from how many high schools did they come each year?

EXPECTED ANSWER:
In 2017, 311 girls from 10 high schools in the Académie de Lyon attended (at the Musée des Confluences). In 2018, 465 girls from 15 high schools attended (at ENS), and about a third of applicants had to be refused.

RAG ANSWER:
- **2017:** 311 high‑school girls attended, coming from 10 different high schools.  
- **2018:** 465 high‑school girls attended, coming from 15 different high schools.


In [82]:
question = df.iloc[7]["question"]

print("QUESTION:")
print(question)

print("\nEXPECTED ANSWER:")
print(df.iloc[7]["answer"])

print("\nRAG ANSWER:")
print(ask_rag(question))

QUESTION:
How many students participated in the Autonomous University of Tlaxcala study of returnees in language programs, what share were in Language Teaching, and what gender split was reported?

EXPECTED ANSWER:
The exploratory study included 26 voluntary participants; 84% were enrolled in the Language Teaching program and 16% in Modern Applied Languages. Of the total, 42.3% (11) were male and 57.7% (15) were female.

RAG ANSWER:
- **Total participants:** 26 students.  
- **Program distribution:** 84 % (≈ 22 students) were enrolled in the **Language Teaching** program, while 16 % (≈ 4 students) were in Modern Applied Languages.  
- **Gender split:** 42.3 % male (11 students) and 57.7 % female (15 students).


In [83]:
question = df.iloc[8]["question"]

print("QUESTION:")
print(question)

print("\nEXPECTED ANSWER:")
print(df.iloc[8]["answer"])

print("\nRAG ANSWER:")
print(ask_rag(question))

QUESTION:
What is the hourly pay for the Island Enterprises Retail LLC Bud Tender part-time position, and what are the position open and close dates?

EXPECTED ANSWER:
The hourly rate is $13.69. The position opens July 12, 2021, and closes July 23, 2021.

RAG ANSWER:
The Island Enterprises Retail LLC Bud Tender part‑time position pays **$13.69 per hour**.  
The position opens on **July 12, 2021** and closes on **July 23, 2021**.


In [84]:
question = df.iloc[9]["question"]

print("QUESTION:")
print(question)

print("\nEXPECTED ANSWER:")
print(df.iloc[9]["answer"])

print("\nRAG ANSWER:")
print(ask_rag(question))

QUESTION:
According to Chicken Farmers of Ontario’s roles and responsibilities document, when does the Board Chair vote at meetings, and what dual legal duties do Directors owe?

EXPECTED ANSWER:
If required, the Chair only votes to break a tie. Each Director participates in the Board’s stewardship by acting honestly and in good faith with a view to the best interests of CFO (fiduciary duty) and by exercising the care, diligence and skill that a reasonably prudent person would exercise in comparable circumstances (duty of care). Directors act in the best interests of CFO as a whole, not any specific constituency.

RAG ANSWER:
**When the Chair votes:**  
The Chair only casts a vote when the Board is tied on a decision (i.e., a 50‑50 split). In all other cases the Chair abstains and lets the other directors decide.

**Dual legal duties owed by Directors:**  
1. **Duty of loyalty** – Directors must act in the best interests of the organization as a whole, not for any individual or group. 

In [85]:
question = df.iloc[10]["question"]

print("QUESTION:")
print(question)

print("\nEXPECTED ANSWER:")
print(df.iloc[10]["answer"])

print("\nRAG ANSWER:")
print(ask_rag(question))

QUESTION:
Who is the employer for the Kickstart Strategy, Project & Communications Assistant role, where is it based, and what Universal Credit eligibility criteria apply?

EXPECTED ANSWER:
The employer is Clarion Housing Group, and the role is based in Norwich, NR7 0WF. Applicants must currently receive Universal Credit, be ages 16–24, and meet suitability and eligibility criteria assessed by their Universal Credit Work Coach.

RAG ANSWER:
- **Employer:** Clarion Housing Group  
- **Location:** Norwich, NR7 0WF  
- **Universal Credit eligibility:** Applicants must be **currently receiving Universal Credit payments**, be **between 16 and 24 years old**, and meet the suitability and eligibility criteria that will be assessed by their Universal Credit Work Coach.


In [86]:
question = df.iloc[11]["question"]

print("QUESTION:")
print(question)

print("\nEXPECTED ANSWER:")
print(df.iloc[11]["answer"])

print("\nRAG ANSWER:")
print(ask_rag(question))

QUESTION:
According to Associate Professor Alexander Johnson, how many people are affected by iron deficiency globally, and where is the first release of iron-biofortified rice likely to occur?

EXPECTED ANSWER:
Iron deficiency affects more than two billion people globally. The first release of iron-biofortified rice will likely occur in Bangladesh.

RAG ANSWER:
According to Associate Professor Alexander Johnson, iron deficiency affects **more than two billion people globally**, and the first release of iron‑biofortified rice is expected to occur in **Bangladesh**.


In [87]:
question = df.iloc[12]["question"]

print("QUESTION:")
print(question)

print("\nEXPECTED ANSWER:")
print(df.iloc[12]["answer"])

print("\nRAG ANSWER:")
print(ask_rag(question))

QUESTION:
According to Graciela Chichilnisky in the 2016 Rozenberg Quarterly interview, roughly how much CO2 do humans put into the atmosphere each year, and how many Global Thermostat carbon-negative plants does she say are needed on average per nation?

EXPECTED ANSWER:
Humans put about 30 gigatons of CO2 into the atmosphere per year. On average, she says about 200 carbon-negative Global Thermostat plants would be needed per nation in the world to clean that CO2 from the atmosphere.

RAG ANSWER:
According to Graciela Chichilnisky in the 2016 Rozenberg Quarterly interview, humans emit roughly **30 gigatons of CO₂ per year**, and she estimates that an average country would need about **200 Global Thermostat carbon‑negative plants**.


In [88]:
question = df.iloc[13]["question"]

print("QUESTION:")
print(question)

print("\nEXPECTED ANSWER:")
print(df.iloc[13]["answer"])

print("\nRAG ANSWER:")
print(ask_rag(question))

QUESTION:
Which bat colony occupies Corkscrew’s parking-lot bat house, and how does the profile describe Florida’s smallest bat species?

EXPECTED ANSWER:
The parking lot bat house is home to a colony of Brazilian Free-tailed Bats (Tadarida brasiliensis); visitors are asked to respect the roost because repeated disturbance could lead to abandonment. Eastern Pipistrelles (Pipistrellus sublavus) are Florida’s smallest bat, weighing about 1/4 ounce, with a body less than two inches long and an eight- to ten-inch wingspan.

RAG ANSWER:
- **Corkscrew’s parking‑lot bat house** is occupied by a colony of **Brazilian Free‑tailed


In [89]:
question = df.iloc[13]["question"]

# Retrieve the top 5 relevant passages
query_embedding = model.encode(
    [question],
    convert_to_numpy=True
).astype("float32")

distances, indices = passage_index.search(query_embedding, 5)

print("QUESTION:")
print(question)

print("\nRETRIEVED PASSAGES:")

for rank, idx in enumerate(indices[0], start=1):
    print(f"\n--- Passage {rank} ---")
    print("Distance:", distances[0][rank - 1])
    print(passages[idx][:1200])

QUESTION:
Which bat colony occupies Corkscrew’s parking-lot bat house, and how does the profile describe Florida’s smallest bat species?

RETRIEVED PASSAGES:

--- Passage 1 ---
Distance: 0.49252266
nguinivores. 

These flying mammals are the best natural insect-control around. 

People are slowly beginning to understand and appreciate these wonderful creatures, but no bats in Florida are completely safe or protected. 

cult, but new technologies are helping us better understand them. However, there still isn’t much information about Florida populations. Education, habitat preservation, and more research are all keys to ensuring the continuation of these beneficial animals. 

Bats that use trees and foliage are losing ground to development, and bats that move into buildings are susceptible 

%IMAGE_DESCRIPTION: A pixelated image of a person wearing a white shirt with their hands raised near their head against a black background.%

At Corkscrew, two bat houses are in place: one by the gu

In [90]:
for i, idx in enumerate(indices[0], start=1):
    passage = passages[idx]

    print(f"\n--- Passage {i} ---")

    for keyword in ["Brazilian Free-tailed", "smallest", "parking lot"]:
        if keyword.lower() in passage.lower():
            print("Found:", keyword)

    print(passage[:1500])


--- Passage 1 ---
Found: Brazilian Free-tailed
Found: parking lot
nguinivores. 

These flying mammals are the best natural insect-control around. 

People are slowly beginning to understand and appreciate these wonderful creatures, but no bats in Florida are completely safe or protected. 

cult, but new technologies are helping us better understand them. However, there still isn’t much information about Florida populations. Education, habitat preservation, and more research are all keys to ensuring the continuation of these beneficial animals. 

Bats that use trees and foliage are losing ground to development, and bats that move into buildings are susceptible 

%IMAGE_DESCRIPTION: A pixelated image of a person wearing a white shirt with their hands raised near their head against a black background.%

At Corkscrew, two bat houses are in place: one by the guest cabin and another in the back of the main parking lot. The parking lot bat house is home to a colony of Brazilian Free-tailed B

In [91]:
keywords = [
    "Brazilian Free-tailed",
    "parking-lot bat house",
    "smallest bat"
]

for keyword in keywords:
    print(f"\n--- Searching: {keyword} ---")

    found = False

    for i, passage in enumerate(passages):
        if keyword.lower() in passage.lower():
            print(f"Passage index: {i}")
            print(passage[:2000])
            found = True

    if not found:
        print("No exact match found.")


--- Searching: Brazilian Free-tailed ---
Passage index: 272
## **Corkscrew’s Bats Friends in the Night** 

|Big Brown Bat<br>_Eptesicus fuscus_|Eastern Pipistrelles<br>_Pipistrelles sublavus_|Northern Yellow Bat<br>_Lasiurus intermedius_|Brazilian Free-tailed Bat<br>_Tadarida brasiliensis_|
|---|---|---|---|
|Rafinesque’s Big-eared Bat<br>_Corynorhinus rafinesquii_|Evening Bat<br>_Nycticeius humeralis_|Seminole Bat<br>_Lasiurus seminolus_|Florida Bonneted Bat<br>_Eumops floridanus_|



As night descends, many animals settle in for the night. Others are just beginning to stir. 

A variety of nocturnal animals call Corkscrew home, including misunderstood and often feared bats. These fascinating animals, the only mammals with true flight, are our friends in the night skies, hunting for insects such as mosquitoes and numerous agricultural pests. Yet little is known about them. 

These brief profiles will introduce you to the mysterious and intriguing world of bats with an emphasis on the 

In [92]:
print(passages[272])

## **Corkscrew’s Bats Friends in the Night** 

|Big Brown Bat<br>_Eptesicus fuscus_|Eastern Pipistrelles<br>_Pipistrelles sublavus_|Northern Yellow Bat<br>_Lasiurus intermedius_|Brazilian Free-tailed Bat<br>_Tadarida brasiliensis_|
|---|---|---|---|
|Rafinesque’s Big-eared Bat<br>_Corynorhinus rafinesquii_|Evening Bat<br>_Nycticeius humeralis_|Seminole Bat<br>_Lasiurus seminolus_|Florida Bonneted Bat<br>_Eumops floridanus_|



As night descends, many animals settle in for the night. Others are just beginning to stir. 

A variety of nocturnal animals call Corkscrew home, including misunderstood and often feared bats. These fascinating animals, the only mammals with true flight, are our friends in the night skies, hunting for insects such as mosquitoes and numerous agricultural pests. Yet little is known about them. 

These brief profiles will introduce you to the mysterious and intriguing world of bats with an emphasis on the species found in the Corkscrew area. 

All bats belong to the

In [93]:
for i, passage in enumerate(passages):
    text = passage.lower()

    if ("parking" in text or "smallest" in text
        or "smallest bat" in text):
        print(f"\n--- Passage {i} ---")
        print(passage[:2500])


--- Passage 276 ---
nguinivores. 

These flying mammals are the best natural insect-control around. 

People are slowly beginning to understand and appreciate these wonderful creatures, but no bats in Florida are completely safe or protected. 

cult, but new technologies are helping us better understand them. However, there still isn’t much information about Florida populations. Education, habitat preservation, and more research are all keys to ensuring the continuation of these beneficial animals. 

Bats that use trees and foliage are losing ground to development, and bats that move into buildings are susceptible 

%IMAGE_DESCRIPTION: A pixelated image of a person wearing a white shirt with their hands raised near their head against a black background.%

At Corkscrew, two bat houses are in place: one by the guest cabin and another in the back of the main parking lot. The parking lot bat house is home to a colony of Brazilian Free-tailed Bats. However, repeated disturbance of this roo

In [94]:
%whos

Variable              Type                   Data/Info
------------------------------------------------------
Groq                  type                   <class 'groq.Groq'>
PdfReader             ABCMeta                <class 'pypdf._reader.PdfReader'>
SentenceTransformer   ABCMeta                <class 'sentence_transfor<...>del.SentenceTransformer'>
api_key               str                    gsk_cl98OuPk43csdzYFbFsPW<...>YN1Nz60sGHzSYI2l32W8omddQ
ask_rag               function               <function ask_rag at 0x7f65b3790b80>
best_context          str                    %IMAGE_DESCRIPTION: Unive<...>ineyards & Wines _._ \n\n
chunk_size            int                    1000
chunk_text            function               <function chunk_text at 0x7f65b380e160>
clean_text            function               <function clean_text at 0x7f65b380e200>
client                Groq                   <groq.Groq object at 0x7f65b0383380>
context               str                    improving the 

In [95]:
question = """
Which bat colony occupies Corkscrew's parking-lot bat house,
and how does the profile describe Florida's smallest bat species?
"""

query_embedding = model.encode(
    [question],
    convert_to_numpy=True
).astype("float32")

distances, indices = passage_index.search(query_embedding, 5)

for rank, idx in enumerate(indices[0], start=1):
    print(f"\n--- Result {rank} | Passage {idx} ---")
    print(passages[idx][:1800])


--- Result 1 | Passage 276 ---
nguinivores. 

These flying mammals are the best natural insect-control around. 

People are slowly beginning to understand and appreciate these wonderful creatures, but no bats in Florida are completely safe or protected. 

cult, but new technologies are helping us better understand them. However, there still isn’t much information about Florida populations. Education, habitat preservation, and more research are all keys to ensuring the continuation of these beneficial animals. 

Bats that use trees and foliage are losing ground to development, and bats that move into buildings are susceptible 

%IMAGE_DESCRIPTION: A pixelated image of a person wearing a white shirt with their hands raised near their head against a black background.%

At Corkscrew, two bat houses are in place: one by the guest cabin and another in the back of the main parking lot. The parking lot bat house is home to a colony of Brazilian Free-tailed Bats. However, repeated disturbance 

In [96]:
for rank, idx in enumerate(indices[0], start=1):
    text = passages[idx]

    print(f"\n--- Result {rank} | Passage {idx} ---")

    if "parking lot bat house" in text.lower():
        print("Found: Parking-lot bat house")

    if "eastern pipistrelle" in text.lower():
        print("Found: Florida's smallest bat")

    print(text[-1000:])


--- Result 1 | Passage 276 ---
Found: Parking-lot bat house
nguinivores. 

These flying mammals are the best natural insect-control around. 

People are slowly beginning to understand and appreciate these wonderful creatures, but no bats in Florida are completely safe or protected. 

cult, but new technologies are helping us better understand them. However, there still isn’t much information about Florida populations. Education, habitat preservation, and more research are all keys to ensuring the continuation of these beneficial animals. 

Bats that use trees and foliage are losing ground to development, and bats that move into buildings are susceptible 

%IMAGE_DESCRIPTION: A pixelated image of a person wearing a white shirt with their hands raised near their head against a black background.%

At Corkscrew, two bat houses are in place: one by the guest cabin and another in the back of the main parking lot. The parking lot bat house is home to a colony of Brazilian Free-tailed Bats. H

In [97]:
print("=== Passage 277: Bat house ===")
print(passages[277])

print("\n=== Passage 283: Smallest bat ===")
print(passages[283])

=== Passage 277: Bat house ===
guest cabin and another in the back of the main parking lot. The parking lot bat house is home to a colony of Brazilian Free-tailed Bats. However, repeated disturbance of this roost could lead to its abandonment, so please respect their home. 

## **Friends in the Night Corkscrew’s Bats** 

## **Big Brown Bat** 

## _Eptesicus fuscus_ 

Big Brown Bats are found throughout North America from Canada to southern Mexico. 

Normally  forest dwellers, they do not hesitate to utilize  attics and crevices in buildings, caves, and rocks for daytime retreats. Favorite roosts are under the  loose bark of dead trees and in cavities of trees. 

Big Brown Bats are just that … big (13-16 inch wingspan) with brown, long fur ranging in color from dark to golden brown. Their undersides are a lighter shade. They have dark brown, almost black ears. Their teeth are impressive because these beneficial bats eat a variety of hard-bodied insects. 

The Big Brown Bat is largely cr

In [98]:
query_embedding = model.encode(
    [question],
    convert_to_numpy=True
).astype("float32")

distances, indices = passage_index.search(query_embedding, 10)

for rank, idx in enumerate(indices[0], start=1):
    print(f"\n--- Result {rank} | Passage {idx} ---")
    print(passages[idx][:1000])


--- Result 1 | Passage 276 ---
nguinivores. 

These flying mammals are the best natural insect-control around. 

People are slowly beginning to understand and appreciate these wonderful creatures, but no bats in Florida are completely safe or protected. 

cult, but new technologies are helping us better understand them. However, there still isn’t much information about Florida populations. Education, habitat preservation, and more research are all keys to ensuring the continuation of these beneficial animals. 

Bats that use trees and foliage are losing ground to development, and bats that move into buildings are susceptible 

%IMAGE_DESCRIPTION: A pixelated image of a person wearing a white shirt with their hands raised near their head against a black background.%

At Corkscrew, two bat houses are in place: one by the guest cabin and another in the back of the main parking lot. The parking lot bat house is home to a colony of Brazilian Free-tailed Bats. However, repeated disturbance 

In [99]:
keywords = [
    "parking lot bat house",
    "Brazilian Free-tailed Bats",
    "Eastern Pipistrelles",
    "Florida's smallest bat"
]

for keyword in keywords:
    print(f"\n--- Searching for: {keyword} ---")

    matches = [
        i for i, passage in enumerate(passages)
        if keyword.lower() in passage.lower()
    ]

    print("Matching passages:", matches[:10])
    print("Total matches:", len(matches))


--- Searching for: parking lot bat house ---
Matching passages: [276, 277]
Total matches: 2

--- Searching for: Brazilian Free-tailed Bats ---
Matching passages: [276, 277, 297, 298]
Total matches: 4

--- Searching for: Eastern Pipistrelles ---
Matching passages: [272, 283, 284, 285]
Total matches: 4

--- Searching for: Florida's smallest bat ---
Matching passages: []
Total matches: 0


In [100]:
for idx in [277, 283, 284, 285]:
    print(f"\n{'=' * 20} Passage {idx} {'=' * 20}")
    print(passages[idx])


==================== Passage 277 ====================
guest cabin and another in the back of the main parking lot. The parking lot bat house is home to a colony of Brazilian Free-tailed Bats. However, repeated disturbance of this roost could lead to its abandonment, so please respect their home. 

## **Friends in the Night Corkscrew’s Bats** 

## **Big Brown Bat** 

## _Eptesicus fuscus_ 

Big Brown Bats are found throughout North America from Canada to southern Mexico. 

Normally  forest dwellers, they do not hesitate to utilize  attics and crevices in buildings, caves, and rocks for daytime retreats. Favorite roosts are under the  loose bark of dead trees and in cavities of trees. 

Big Brown Bats are just that … big (13-16 inch wingspan) with brown, long fur ranging in color from dark to golden brown. Their undersides are a lighter shade. They have dark brown, almost black ears. Their teeth are impressive because these beneficial bats eat a variety of hard-bodied insects. 

The Big

In [101]:
for idx in [283, 284, 285]:
    print(f"\n--- Passage {idx} ---")
    print(passages[idx][-1200:])


--- Passage 283 ---
ee months. They can live 

Biologists age bats by looking at the finger bones in the outstretched wings over a bright light. Juvenile bats’ bones have clear spaces between the joints; adult bats have ossified joints. 

## **Friends in the Night Corkscrew’s Bats** 

## **Eastern Pipistrelles** 

## _Pipistrellus sublavus_ 

Eastern Pipistrelles is Florida’s smallest bat, weighing in at a mere 1/4 ounce. The body is less than two inches long, with an eight to ten inch wingspan. They are solitary except at mating and in small nursery groups. 

Eastern Pipistrelles are some of the earliest bats to emerge in the evening and are often be seen in the ambient light as they move to foraging grounds. They are relatively slow and erratic in flight, often fluttering and flitting along watercourses or over pastures and woodlands much like large moths. 

They can catch prey at a rate of one insect every two seconds, increasing body mass as much as 25 percent in just thirty minut

In [102]:

query = "Eastern Pipistrelles Florida's smallest bat"

query_embedding = model.encode(
    [query],
    convert_to_numpy=True
).astype("float32")

distances, indices = passage_index.search(query_embedding, 5)

for rank, idx in enumerate(indices[0], start=1):
    print(f"\n--- Result {rank} | Passage {idx} ---")
    print(passages[idx][:1000])



--- Result 1 | Passage 283 ---
ee months. They can live 

Biologists age bats by looking at the finger bones in the outstretched wings over a bright light. Juvenile bats’ bones have clear spaces between the joints; adult bats have ossified joints. 

## **Friends in the Night Corkscrew’s Bats** 

## **Eastern Pipistrelles** 

## _Pipistrellus sublavus_ 

Eastern Pipistrelles is Florida’s smallest bat, weighing in at a mere 1/4 ounce. The body is less than two inches long, with an eight to ten inch wingspan. They are solitary except at mating and in small nursery groups. 

Eastern Pipistrelles are some of the earliest bats to emerge in the evening and are often be seen in the ambient light as they move to foraging grounds. They are relatively slow and erratic in flight, often fluttering and flitting along watercourses or over pastures and woodlands much like large moths. 

They can catch prey at a rate of one insect every two seconds, increasing body mass as much as 25 percent in just t

In [103]:

keywords = ["parking-lot bat house", "Eastern Pipistrelle"]

for keyword in keywords:
    print(f"\n===== Searching: {keyword} =====")

    matches = [
        (i, passage)
        for i, passage in enumerate(passages)
        if keyword.lower() in passage.lower()
    ]

    if matches:
        for i, passage in matches[:3]:
            print(f"\nPassage {i}:")
            print(passage[:1200])
    else:
        print("No exact match found.")



===== Searching: parking-lot bat house =====
No exact match found.

===== Searching: Eastern Pipistrelle =====

Passage 272:
## **Corkscrew’s Bats Friends in the Night** 

|Big Brown Bat<br>_Eptesicus fuscus_|Eastern Pipistrelles<br>_Pipistrelles sublavus_|Northern Yellow Bat<br>_Lasiurus intermedius_|Brazilian Free-tailed Bat<br>_Tadarida brasiliensis_|
|---|---|---|---|
|Rafinesque’s Big-eared Bat<br>_Corynorhinus rafinesquii_|Evening Bat<br>_Nycticeius humeralis_|Seminole Bat<br>_Lasiurus seminolus_|Florida Bonneted Bat<br>_Eumops floridanus_|



As night descends, many animals settle in for the night. Others are just beginning to stir. 

A variety of nocturnal animals call Corkscrew home, including misunderstood and often feared bats. These fascinating animals, the only mammals with true flight, are our friends in the night skies, hunting for insects such as mosquitoes and numerous agricultural pests. Yet little is known about them. 

These brief profiles will introduce you to the

In [104]:

keywords = ["parking lot", "bat house", "smallest bat", "Pipistrellus subflavus"]

for keyword in keywords:
    print(f"\n{'='*20} {keyword} {'='*20}")

    count = 0
    for i, passage in enumerate(passages):
        if keyword.lower() in passage.lower():
            print(f"\nPassage {i}:")
            print(passage[:1500])
            count += 1

            if count == 3:
                break

    if count == 0:
        print("No match found.")



==================== parking lot ====================

Passage 276:
nguinivores. 

These flying mammals are the best natural insect-control around. 

People are slowly beginning to understand and appreciate these wonderful creatures, but no bats in Florida are completely safe or protected. 

cult, but new technologies are helping us better understand them. However, there still isn’t much information about Florida populations. Education, habitat preservation, and more research are all keys to ensuring the continuation of these beneficial animals. 

Bats that use trees and foliage are losing ground to development, and bats that move into buildings are susceptible 

%IMAGE_DESCRIPTION: A pixelated image of a person wearing a white shirt with their hands raised near their head against a black background.%

At Corkscrew, two bat houses are in place: one by the guest cabin and another in the back of the main parking lot. The parking lot bat house is home to a colony of Brazilian Free-tailed

In [106]:

for i, passage in enumerate(passages):
    if "Pipistrellus subflavus" in passage:
        print(f"Passage {i}")
        print(passage)
        print("-" * 60)

In [107]:

print("Total passages:", len(passages))

for i, passage in enumerate(passages):
    if "pipistrelle" in passage.lower():
        print(f"\nPassage {i}:")
        print(passage[:2000])


Total passages: 2749

Passage 272:
## **Corkscrew’s Bats Friends in the Night** 

|Big Brown Bat<br>_Eptesicus fuscus_|Eastern Pipistrelles<br>_Pipistrelles sublavus_|Northern Yellow Bat<br>_Lasiurus intermedius_|Brazilian Free-tailed Bat<br>_Tadarida brasiliensis_|
|---|---|---|---|
|Rafinesque’s Big-eared Bat<br>_Corynorhinus rafinesquii_|Evening Bat<br>_Nycticeius humeralis_|Seminole Bat<br>_Lasiurus seminolus_|Florida Bonneted Bat<br>_Eumops floridanus_|



As night descends, many animals settle in for the night. Others are just beginning to stir. 

A variety of nocturnal animals call Corkscrew home, including misunderstood and often feared bats. These fascinating animals, the only mammals with true flight, are our friends in the night skies, hunting for insects such as mosquitoes and numerous agricultural pests. Yet little is known about them. 

These brief profiles will introduce you to the mysterious and intriguing world of bats with an emphasis on the species found in the Corks

In [108]:

print("PASSAGE 283")
print(passages[283])

print("\n" + "=" * 50)
print("PASSAGE 277")
print(passages[277])


PASSAGE 283
ee months. They can live 

Biologists age bats by looking at the finger bones in the outstretched wings over a bright light. Juvenile bats’ bones have clear spaces between the joints; adult bats have ossified joints. 

## **Friends in the Night Corkscrew’s Bats** 

## **Eastern Pipistrelles** 

## _Pipistrellus sublavus_ 

Eastern Pipistrelles is Florida’s smallest bat, weighing in at a mere 1/4 ounce. The body is less than two inches long, with an eight to ten inch wingspan. They are solitary except at mating and in small nursery groups. 

Eastern Pipistrelles are some of the earliest bats to emerge in the evening and are often be seen in the ambient light as they move to foraging grounds. They are relatively slow and erratic in flight, often fluttering and flitting along watercourses or over pastures and woodlands much like large moths. 

They can catch prey at a rate of one insect every two seconds, increasing body mass as much as 25 percent in just thirty minutes. Typic

In [109]:

query = """Which bat colony occupies Corkscrew's parking-lot
bat house, and how does the profile describe Florida's
smallest bat?"""

query_embedding = model.encode(
    [query],
    convert_to_numpy=True
).astype("float32")

distances, indices = passage_index.search(query_embedding, 5)

print("Question:", query)

for rank, idx in enumerate(indices[0], start=1):
    print(f"\n--- Retrieved Result {rank} | Passage {idx} ---")
    print(passages[idx][:1500])


Question: Which bat colony occupies Corkscrew's parking-lot
bat house, and how does the profile describe Florida's
smallest bat?

--- Retrieved Result 1 | Passage 276 ---
nguinivores. 

These flying mammals are the best natural insect-control around. 

People are slowly beginning to understand and appreciate these wonderful creatures, but no bats in Florida are completely safe or protected. 

cult, but new technologies are helping us better understand them. However, there still isn’t much information about Florida populations. Education, habitat preservation, and more research are all keys to ensuring the continuation of these beneficial animals. 

Bats that use trees and foliage are losing ground to development, and bats that move into buildings are susceptible 

%IMAGE_DESCRIPTION: A pixelated image of a person wearing a white shirt with their hands raised near their head against a black background.%

At Corkscrew, two bat houses are in place: one by the guest cabin and another in th

In [110]:

query = """Which bat colony occupies Corkscrew's parking-lot
bat house, and how does the profile describe Florida's
smallest bat?"""

# Find passages using relevant keywords
keywords = ["parking lot bat house", "Eastern Pipistrelles"]

matched_passages = []

for i, passage in enumerate(passages):
    text = passage.lower()

    if (
        "parking lot bat house" in text
        or "eastern pipistrelles is florida's smallest bat" in text
        or "eastern pipistrelles is florida’s smallest bat" in text
    ):
        matched_passages.append((i, passage))

print("Matching passages found:", len(matched_passages))

for i, passage in matched_passages:
    print(f"\n--- Passage {i} ---")
    print(passage[:1200])


Matching passages found: 3

--- Passage 276 ---
nguinivores. 

These flying mammals are the best natural insect-control around. 

People are slowly beginning to understand and appreciate these wonderful creatures, but no bats in Florida are completely safe or protected. 

cult, but new technologies are helping us better understand them. However, there still isn’t much information about Florida populations. Education, habitat preservation, and more research are all keys to ensuring the continuation of these beneficial animals. 

Bats that use trees and foliage are losing ground to development, and bats that move into buildings are susceptible 

%IMAGE_DESCRIPTION: A pixelated image of a person wearing a white shirt with their hands raised near their head against a black background.%

At Corkscrew, two bat houses are in place: one by the guest cabin and another in the back of the main parking lot. The parking lot bat house is home to a colony of Brazilian Free-tailed Bats. However, repea

In [111]:

context_passages = []

for i, passage in enumerate(passages):
    text = passage.lower()

    if (
        "parking lot bat house" in text
        or "eastern pipistrelles is florida" in text
    ):
        context_passages.append((i, passage))

context = "\n\n".join(
    f"Passage {i}:\n{passage}"
    for i, passage in context_passages
)

print("Passages included:", len(context_passages))
print(context[:5000])


Passages included: 3
Passage 276:
nguinivores. 

These flying mammals are the best natural insect-control around. 

People are slowly beginning to understand and appreciate these wonderful creatures, but no bats in Florida are completely safe or protected. 

cult, but new technologies are helping us better understand them. However, there still isn’t much information about Florida populations. Education, habitat preservation, and more research are all keys to ensuring the continuation of these beneficial animals. 

Bats that use trees and foliage are losing ground to development, and bats that move into buildings are susceptible 

%IMAGE_DESCRIPTION: A pixelated image of a person wearing a white shirt with their hands raised near their head against a black background.%

At Corkscrew, two bat houses are in place: one by the guest cabin and another in the back of the main parking lot. The parking lot bat house is home to a colony of Brazilian Free-tailed Bats. However, repeated disturbanc

In [112]:

selected_ids = [276, 277, 283]

context = "\n\n".join(
    f"Passage {i}:\n{passages[i]}"
    for i in selected_ids
)

print(context)


Passage 276:
nguinivores. 

These flying mammals are the best natural insect-control around. 

People are slowly beginning to understand and appreciate these wonderful creatures, but no bats in Florida are completely safe or protected. 

cult, but new technologies are helping us better understand them. However, there still isn’t much information about Florida populations. Education, habitat preservation, and more research are all keys to ensuring the continuation of these beneficial animals. 

Bats that use trees and foliage are losing ground to development, and bats that move into buildings are susceptible 

%IMAGE_DESCRIPTION: A pixelated image of a person wearing a white shirt with their hands raised near their head against a black background.%

At Corkscrew, two bat houses are in place: one by the guest cabin and another in the back of the main parking lot. The parking lot bat house is home to a colony of Brazilian Free-tailed Bats. However, repeated disturbance of this roost could

In [113]:

print("Eastern Pipistrelle description:")
print(passages[283][0:1500])


Eastern Pipistrelle description:
ee months. They can live 

Biologists age bats by looking at the finger bones in the outstretched wings over a bright light. Juvenile bats’ bones have clear spaces between the joints; adult bats have ossified joints. 

## **Friends in the Night Corkscrew’s Bats** 

## **Eastern Pipistrelles** 

## _Pipistrellus sublavus_ 

Eastern Pipistrelles is Florida’s smallest bat, weighing in at a mere 1/4 ounce. The body is less than two inches long, with an eight to ten inch wingspan. They are solitary except at mating and in small nursery groups. 

Eastern Pipistrelles are some of the earliest bats to emerge in the evening and are often be seen in the ambient light as they move to foraging grounds. They are relatively slow and erratic in flight, often fluttering and flitting along watercourses or over pastures and woodlands much like large moths. 

They can catch prey at a rate of one insect every two seconds, increasing body mass as much as 25 percent in just 

In [114]:

question = """Which bat colony occupies Corkscrew's parking-lot
bat house, and how does the profile describe Florida's
smallest bat?"""

prompt = f"""
Answer the question using only the context below.
If the context does not contain an answer, say so.
Do not add unsupported facts.

Context:
{context}

Question:
{question}

Answer:
"""

print(prompt[:5000])



Answer the question using only the context below.
If the context does not contain an answer, say so.
Do not add unsupported facts.

Context:
Passage 276:
nguinivores. 

These flying mammals are the best natural insect-control around. 

People are slowly beginning to understand and appreciate these wonderful creatures, but no bats in Florida are completely safe or protected. 

cult, but new technologies are helping us better understand them. However, there still isn’t much information about Florida populations. Education, habitat preservation, and more research are all keys to ensuring the continuation of these beneficial animals. 

Bats that use trees and foliage are losing ground to development, and bats that move into buildings are susceptible 

%IMAGE_DESCRIPTION: A pixelated image of a person wearing a white shirt with their hands raised near their head against a black background.%

At Corkscrew, two bat houses are in place: one by the guest cabin and another in the back of the ma

In [115]:

print("Brazilian Free-tailed Bats found:",
      "Brazilian Free-tailed Bats" in context)

print("Eastern Pipistrelle found:",
      "Eastern Pipistrelle" in context)

print("\nEastern Pipistrelle section:")
for line in context.splitlines():
    if "smallest bat" in line.lower() or "ounce" in line.lower():
        print(line)


Brazilian Free-tailed Bats found: True
Eastern Pipistrelle found: True

Eastern Pipistrelle section:
Eastern Pipistrelles is Florida’s smallest bat, weighing in at a mere 1/4 ounce. The body is less than two inches long, with an eight to ten inch wingspan. They are solitary except at mating and in small nursery groups. 


In [118]:

try:
    models = client.models.list()

    print("Models available to your Groq client:\n")
    for item in models.data:
        print(item.id)

except Exception as e:
    print("Error type:", type(e).__name__)
    print("Error details:", str(e))


Models available to your Groq client:

qwen/qwen3.8-27b
canopylabs/orpheus-v1-english
allam-2-7b
whisper-large-v3-turbo
openai/gpt-oss-120b
meta-llama/llama-prompt-guard-2-22m
openai/gpt-oss-20b
whisper-large-v3
canopylabs/orpheus-arabic-saudi
openai/gpt-oss-safeguard-20b
meta-llama/llama-prompt-guard-2-86m


In [119]:

response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {
            "role": "system",
            "content": "Answer using only the provided context. Do not invent facts."
        },
        {
            "role": "user",
            "content": prompt
        }
    ],
    temperature=0
)

print(response.choices[0].message.content)


The parking‑lot bat house at Corkscrew is home to a colony of **Brazilian Free‑tailed Bats**.  

Florida’s smallest bat, described in the profile, is the **Eastern Pipistrelle**. It is a very tiny bat—about **¼ ounce in weight**, **under two inches long** with an **8‑ to 10‑inch wingspan**. It is usually solitary (except when mating or in small nursery groups), is one of the earliest bats to emerge at dusk, and flies slowly and erratically, fluttering like a large moth.


In [120]:

!pip install -q pypdf


In [124]:

import os

pdf_files = [
    file for file in os.listdir("/content")
    if file.lower().endswith(".pdf")
]

print("PDF files found:")
for file in pdf_files:
    print(file)


PDF files found:
RAG_COLLAGE_DOCUMENTS.pdf


In [125]:

from pypdf import PdfReader

pdf_path = "/content/RAG_COLLAGE_DOCUMENTS.pdf"

reader = PdfReader(pdf_path)

print("Total pages:", len(reader.pages))

pdf_text = ""

for page in reader.pages:
    text = page.extract_text()
    if text:
        pdf_text += text + "\n"

print("\nExtracted text length:", len(pdf_text))
print("\nFirst 3000 characters:\n")
print(pdf_text[:3000])


Total pages: 41

Extracted text length: 106960

First 3000 characters:

 Visvesvaraya Technological University, Belagavi 
 
 
 
 
 
 
 
 
 
 
 
VTU Regulations Governing the UG Programmes  
B.E. and B.Tech. - 2025 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
                                                                2025  
22052026
           Visvesvaraya Technological University, Belagavi 
          VTU Regulations Governing the UG Programmes B.E. and B.Tech. – 2025  
- 2 - 
 
Contents 
 
VTU Regulations Governing the UG Programmes B.E. and B.Tech. - 2025 
Preamble to Regulations    
Definition of Key Words   
25OB1.0 Title, Duration and Academic Calandar of the Programme 
25OB1.1 Programme of Study 
25OB1.2 Duration of the B.E./ B.Tech. programme 
25OB1.3 Academic Calendar 
25OB2.0 Academic Eligibility for Admission 
25OB2.1 Admission to First Year of the Programme 
25OB2.2 Admission to the Second Year of the Programme (Lateral Entry) 
25OB2.3 Qualification Earned from Foreign Countries   
25

In [126]:

chunk_size = 1000
overlap = 200

chunks = []

start = 0

while start < len(pdf_text):
    end = start + chunk_size
    chunk = pdf_text[start:end]

    if chunk.strip():
        chunks.append(chunk)

    start += chunk_size - overlap

print("Total chunks created:", len(chunks))
print("\nFirst chunk:\n")
print(chunks[0])


Total chunks created: 134

First chunk:

 Visvesvaraya Technological University, Belagavi 
 
 
 
 
 
 
 
 
 
 
 
VTU Regulations Governing the UG Programmes  
B.E. and B.Tech. - 2025 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
                                                                2025  
22052026
           Visvesvaraya Technological University, Belagavi 
          VTU Regulations Governing the UG Programmes B.E. and B.Tech. – 2025  
- 2 - 
 
Contents 
 
VTU Regulations Governing the UG Programmes B.E. and B.Tech. - 2025 
Preamble to Regulations    
Definition of Key Words   
25OB1.0 Title, Duration and Academic Calandar of the Programme 
25OB1.1 Programme of Study 
25OB1.2 Duration of the B.E./ B.Tech. programme 
25OB1.3 Academic Calendar 
25OB2.0 Academic Eligibility for Admission 
25OB2.1 Admission to First Year of the Programme 
25OB2.2 Admission to the Second Year of the Programme (Lateral Entry) 
25OB2.3 Qualification Earned from Foreign Countries   
25OB2.4 Additional Requirements o

In [127]:

from sentence_transformers import SentenceTransformer

embedding_model_vtu = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

chunk_embeddings = embedding_model_vtu.encode(
    chunks,
    convert_to_numpy=True,
    show_progress_bar=True
)

print("Number of chunks:", len(chunks))
print("Embedding shape:", chunk_embeddings.shape)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Number of chunks: 134
Embedding shape: (134, 384)


In [128]:

import faiss
import numpy as np

chunk_embeddings = np.asarray(
    chunk_embeddings,
    dtype="float32"
)

vtu_index = faiss.IndexFlatL2(
    chunk_embeddings.shape[1]
)

vtu_index.add(chunk_embeddings)

print("VTU index created successfully!")
print("Total vectors in index:", vtu_index.ntotal)


VTU index created successfully!
Total vectors in index: 134


In [129]:

question = "What is the maximum duration allowed to complete the B.E. programme?"

question_embedding = embedding_model_vtu.encode(
    [question],
    convert_to_numpy=True
).astype("float32")

distances, indices = vtu_index.search(
    question_embedding,
    3
)

for rank, idx in enumerate(indices[0], start=1):
    print(f"\n--- Result {rank} | Chunk {idx} ---")
    print(chunks[idx][:1000])



--- Result 1 | Chunk 34 ---
s Governing the UG Programmes B.E. and B.Tech. – 2025  
- 13 - 
 
In the event of any dispute regarding the equivalence of qualifications obtained from 
foreign countries, the decision of the University in this regard shall be final and binding 
on all concerned.   
25OB2.4 Additional Requirements of Lateral Entry Candidates    
All Diploma and B.Sc. degree holders seeking admission through lateral entry shall be 
required to successfully complete any additional courses specified by the University 
before the completion of the degree programme.   
 
25OB3.0 Maximum Duration of B.E./B.Tech. Programme and Time Limit for 
Completion  
(a)The duration of the Under Graduate Degree programme, namely B.E./B. Tech., shall be 
four (4) calendar years for students admitted to the first year of the programme with an 
educational qualification of PUC/12 th standard or equivalent. Such students shall 
complete the programme within eight (8) calendar years from the year 

In [130]:

question = "What is the maximum duration allowed to complete the B.E. programme?"

context = "\n\n".join(
    chunks[idx] for idx in indices[0]
)

prompt = f"""
Answer the question using only the VTU regulations
provided in the context.

If the information is missing, say so.
Keep the answer simple and mention the relevant
regulation number when available.

Context:
{context}

Question:
{question}
"""

response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {
            "role": "system",
            "content": "Answer accurately using only the supplied context."
        },
        {
            "role": "user",
            "content": prompt
        }
    ],
    temperature=0
)

print("Question:", question)
print("\nAnswer:")
print(response.choices[0].message.content)


Question: What is the maximum duration allowed to complete the B.E. programme?

Answer:
**Maximum time to finish the B.E. programme**

- **Regular first‑year entrants:** must complete the degree **within 8 calendar years** of their first admission.  
  *Regulation 25OB3.0 (a)*  

- **Lateral‑entry (second‑year) entrants:** must complete the degree **within 6 calendar years** of their first admission.  
  *Regulation 25OB2.0 (b)*  

Thus, the longest allowable period is **8 years** (for students who start in the first year).


In [131]:

question = "How many semesters are there in the B.E. programme?"

question_embedding = embedding_model_vtu.encode(
    [question],
    convert_to_numpy=True
).astype("float32")

distances, indices = vtu_index.search(
    question_embedding, 3
)

context = "\n\n".join(
    chunks[idx] for idx in indices[0]
)

prompt = f"""
Answer the question using only the VTU regulations below.
If the answer is missing, say so.

Context:
{context}

Question: {question}
"""

response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {
            "role": "system",
            "content": "Answer accurately using only the supplied context."
        },
        {"role": "user", "content": prompt}
    ],
    temperature=0
)

print("Question:", question)
print("\nAnswer:", response.choices[0].message.content)
print("\nRetrieved context:\n", context[:1500])


Question: How many semesters are there in the B.E. programme?

Answer: The regular B.E. programme is organised into **eight (08) semesters** (plus four summer semesters). 【VTU regulation 25OB2.0 / 25OB1.2】.

Retrieved context:
 ech. programme 
The duration of the B.E./ B.Tech. programme for shall be four (04) academic years , 
organised in to eight (08) semesters with four (04) summer semesters in the case of 
candidates admitted to the first year of the programme and three (03) academic years, 
organised in six (06) semesters with three (03) summer semesters in the case of lateral 
entry candidates.  
25OB1.3 Academic Calendar 
The annual academic calendar of events for the programme of study, as approved by the 
University, shall be notified before the commencement of each academic year. It shall be 
mandatory for faculty members/students to strictly adhere to the Academic Calendar for 
the completion of academic activities. The Academic Calendar shall be notified on the 
University 

In [132]:

question = "What is the hostel fee for CSE students at Bangalore Technological Institute?"

question_embedding = embedding_model_vtu.encode(
    [question],
    convert_to_numpy=True
).astype("float32")

distances, indices = vtu_index.search(
    question_embedding, 3
)

context = "\n\n".join(
    chunks[idx] for idx in indices[0]
)

prompt = f"""
Answer using only the supplied VTU regulations.
If the document does not contain the answer, clearly
say that the information is not available.
Do not guess or invent fees.

Context:
{context}

Question:
{question}
"""

response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {
            "role": "system",
            "content": "Use only the supplied context. Never invent missing information."
        },
        {"role": "user", "content": prompt}
    ],
    temperature=0
)

print("Question:", question)
print("\nAnswer:", response.choices[0].message.content)


Question: What is the hostel fee for CSE students at Bangalore Technological Institute?

Answer: The provided VTU regulations do not contain any information about hostel fees for CSE students at Bangalore Technological Institute. Therefore, the requested information is not available in the supplied document.
